# ED Pipeline v8 — Self‑contained (embeds canonical), v6 Tracker Schema, Hard Phase‑1 Gate
Embedded `ed_pipeline_v8.py` SHA‑256: `8a54a37cbeb4d31ac543d0003964b10a9bf9a073ccb47bc41e37e3625af7e20a`


In [ ]:
# === Cell -1: Embed canonical ed_pipeline_v8.py into working dir ===
from pathlib import Path
import hashlib
EMBED_PATH = Path('./ed_pipeline_v8.py')
ED_V8_SOURCE = "{\n \"cells\": [\n  {\n   \"cell_type\": \"markdown\",\n   \"id\": \"header\",\n   \"metadata\": {},\n   \"source\": [\n    \"# ED Pipeline v8 - Complete Operational + Clinical Integration\\n\",\n    \"\\n\",\n    \"**Priority**: Operational tools first, clinical algorithms second\\n\",\n    \"\\n\",\n    \"**Phase 1 (Core)**: Equipment tracking, SOP access, lingering patient monitoring\\n\",\n    \"**Phase 2 (Enhanced)**: HL7v2 processing, risk scores, STEMI protocols, audit framework\\n\",\n    \"\\n\",\n    \"**Contract**: Phase 1 tools remain primary interface, Phase 2 optional behind RUN_PIPELINE flag\"\n   ]\n  },\n  {\n   \"cell_type\": \"code\",\n   \"execution_count\": null,\n   \"id\": \"bootstrap\",\n   \"metadata\": {},\n   \"outputs\": [],\n   \"source\": [\n    \"# PHASE 1: OPERATIONAL INFRASTRUCTURE (PRESERVED FROM v6 BASELINE)\\n\",\n    \"import os, sys\\n\",\n    \"from pathlib import Path\\n\",\n    \"if \\\"/mnt/data\\\" not in sys.path: sys.path.insert(0, \\\"/mnt/data\\\")\\n\",\n    \"try:\\n\",\n    \"    CONFIG\\n\",\n    \"except NameError:\\n\",\n    \"    DATA_ROOT = os.environ.get(\\\"DATA_ROOT\\\", \\\"/mnt/data\\\")\\n\",\n    \"    CONFIG = {\\\"DATA_ROOT\\\": DATA_ROOT}\\n\",\n    \"defaults = {\\n\",\n    \"    \\\"EQUIPMENT_STATUS_PATH\\\": str(Path(CONFIG.get(\\\"DATA_ROOT\\\",\\\"/mnt/data\\\")) / \\\"equipment_status.csv\\\"),\\n\",\n    \"    \\\"EQUIPMENT_MOVES_LOG_PATH\\\": str(Path(CONFIG.get(\\\"DATA_ROOT\\\",\\\"/mnt/data\\\")) / \\\"equipment_moves.csv\\\"),\\n\",\n    \"    \\\"SOP_REGISTRY_PATH\\\": str(Path(CONFIG.get(\\\"DATA_ROOT\\\",\\\"/mnt/data\\\")) / \\\"sop_registry.csv\\\"),\\n\",\n    \"    \\\"QR_OUTPUT_DIR\\\": str(Path(CONFIG.get(\\\"DATA_ROOT\\\",\\\"/mnt/data\\\")) / \\\"qr\\\"),\\n\",\n    \"    \\\"EVENT_LOG_PATH\\\": str(Path(CONFIG.get(\\\"DATA_ROOT\\\",\\\"/mnt/data\\\")) / \\\"event_log.jsonl\\\"),\\n\",\n    \"    \\\"RUN_UI\\\": False,\\n\",\n    \"    \\\"RUN_PIPELINE\\\": False,  # Phase 2 disabled by default per requirements\\n\",\n    \"}\\n\",\n    \"CONFIG.update({k: CONFIG.get(k, v) for k, v in defaults.items()})\\n\",\n    \"RUN_UI = CONFIG[\\\"RUN_UI\\\"]; RUN_PIPELINE = CONFIG[\\\"RUN_PIPELINE\\\"]\\n\",\n    \"for k in [\\\"QR_OUTPUT_DIR\\\",\\\"EVENT_LOG_PATH\\\",\\\"SOP_REGISTRY_PATH\\\",\\\"EQUIPMENT_STATUS_PATH\\\",\\\"EQUIPMENT_MOVES_LOG_PATH\\\"]:\\n\",\n    \"    p = Path(CONFIG[k]); (p.parent if p.suffix else p).mkdir(parents=True, exist_ok=True)\\n\",\n    \"print(\\\"\u2705 Phase 1 bootstrap ready (operational tools prioritized)\\\")\"\n   ]\n  },\n  {\n   \"cell_type\": \"code\",\n   \"execution_count\": null,\n   \"id\": \"imports\",\n   \"metadata\": {},\n   \"outputs\": [],\n   \"source\": [\n    \"# CORE WORKFLOW STATE (CONTRACT PRESERVED)\\n\",\n    \"from __future__ import annotations\\n\",\n    \"from dataclasses import dataclass, field\\n\",\n    \"from typing import Optional, Dict, Any, List\\n\",\n    \"import pandas as pd\"\n   ]\n  },\n  {\n   \"cell_type\": \"code\",\n   \"execution_count\": null,\n   \"id\": \"workflow_state\",\n   \"metadata\": {},\n   \"outputs\": [],\n   \"source\": [\n    \"# WORKFLOW STATE + CLINICAL SKILLS (CONTRACT PRESERVED)\\n\",\n    \"@dataclass\\n\",\n    \"class WorkflowState:\\n\",\n    \"    encounter_id: Optional[str] = None\\n\",\n    \"    patient_id: Optional[str] = None\\n\",\n    \"    pending_orders: set = field(default_factory=set)\\n\",\n    \"    completed_studies: set = field(default_factory=set)\\n\",\n    \"    active_consults: set = field(default_factory=set)\\n\",\n    \"    last_vitals_ts: Optional[pd.Timestamp] = None\\n\",\n    \"    chest_pain: bool = False\\n\",\n    \"    trauma: bool = False\\n\",\n    \"    # context\\n\",\n    \"    backlog_ct: int = 0\\n\",\n    \"    backlog_lab: int = 0\\n\",\n    \"    backlog_ecg: int = 0\\n\",\n    \"    hour: int = 12\\n\",\n    \"    role: str = \\\"nurse\\\"\\n\",\n    \"\\n\",\n    \"def skill_need_ecg(state: WorkflowState) -> Optional[Dict[str,Any]]:\\n\",\n    \"    if state.chest_pain and (\\\"ORDER_ECG\\\" not in state.pending_orders) and (\\\"ORDER_ECG\\\" not in state.completed_studies):\\n\",\n    \"        return {\\\"action\\\":\\\"ORDER_ECG\\\", \\\"reason\\\":\\\"Chest pain without ECG\\\", \\\"urgency\\\":\\\"high\\\"}\\n\",\n    \"    return None\\n\",\n    \"\\n\",\n    \"def skill_abnormal_ecg_no_consult(state: WorkflowState) -> Optional[Dict[str,Any]]:\\n\",\n    \"    if (\\\"ORDER_ECG\\\" in state.completed_studies) and (\\\"ECG_ABNORMAL\\\" in state.completed_studies) and (\\\"CARDIOLOGY\\\" not in state.active_consults):\\n\",\n    \"        return {\\\"action\\\":\\\"PAGE_CARDIOLOGY\\\", \\\"reason\\\":\\\"Abnormal ECG without consult\\\", \\\"urgency\\\":\\\"high\\\"}\\n\",\n    \"    return None\\n\",\n    \"\\n\",\n    \"def skill_ct_delayed(state: WorkflowState) -> Optional[Dict[str,Any]]:\\n\",\n    \"    if (\\\"ORDER_CT\\\" in state.pending_orders) and (\\\"CT_RESULT\\\" not in state.completed_studies):\\n\",\n    \"        return {\\\"action\\\":\\\"FOLLOW_UP_IMAGING\\\", \\\"reason\\\":\\\"CT pending > 60m\\\", \\\"urgency\\\":\\\"medium\\\"}\\n\",\n    \"    return None\\n\",\n    \"\\n\",\n    \"def skill_pending_labs_deteriorating(state: WorkflowState) -> Optional[Dict[str,Any]]:\\n\",\n    \"    if ((\\\"LAB_TROPONIN\\\" in state.pending_orders) or (\\\"LAB_PANEL\\\" in state.pending_orders)) and (\\\"Deteriorating\\\" in state.completed_studies):\\n\",\n    \"        return {\\\"action\\\":\\\"EXPEDITE_LABS\\\", \\\"reason\\\":\\\"Pending labs + deterioration\\\", \\\"urgency\\\":\\\"high\\\"}\\n\",\n    \"    return None\\n\",\n    \"\\n\",\n    \"# V8 ENHANCEMENT: Add Phase 1 operational skills\\n\",\n    \"def skill_equipment_overdue(state: WorkflowState) -> Optional[Dict[str,Any]]:\\n\",\n    \"    \\\"\\\"\\\"Operational skill: Check for overdue equipment.\\\"\\\"\\\"\\n\",\n    \"    # This would integrate with TrackerService in real implementation\\n\",\n    \"    return {\\\"action\\\":\\\"CHECK_EQUIPMENT_STATUS\\\", \\\"reason\\\":\\\"Equipment location check overdue\\\", \\\"urgency\\\":\\\"low\\\"}\\n\",\n    \"\\n\",\n    \"def skill_sop_access_needed(state: WorkflowState) -> Optional[Dict[str,Any]]:\\n\",\n    \"    \\\"\\\"\\\"Operational skill: Suggest SOP access for chest pain.\\\"\\\"\\\"\\n\",\n    \"    if state.chest_pain:\\n\",\n    \"        return {\\\"action\\\":\\\"ACCESS_CHEST_PAIN_SOP\\\", \\\"reason\\\":\\\"Chest pain protocol needed\\\", \\\"urgency\\\":\\\"medium\\\"}\\n\",\n    \"    return None\\n\",\n    \"\\n\",\n    \"SKILLS = [\\n\",\n    \"    skill_need_ecg,\\n\",\n    \"    skill_abnormal_ecg_no_consult,\\n\",\n    \"    skill_ct_delayed,\\n\",\n    \"    skill_pending_labs_deteriorating,\\n\",\n    \"    skill_equipment_overdue,  # V8: Operational\\n\",\n    \"    skill_sop_access_needed,  # V8: Operational\\n\",\n    \"]\\n\",\n    \"\\n\",\n    \"def generate_candidates(state: WorkflowState) -> List[Dict[str,Any]]:\\n\",\n    \"    out = []\\n\",\n    \"    for s in SKILLS:\\n\",\n    \"        r = s(state)\\n\",\n    \"        if r: out.append(r)\\n\",\n    \"    return out[:5]\"\n   ]\n  },\n  {\n   \"cell_type\": \"code\",\n   \"execution_count\": null,\n   \"id\": \"tiny_critics\",\n   \"metadata\": {},\n   \"outputs\": [],\n   \"source\": [\n    \"# TINY CRITICS (CONTRACT PRESERVED)\\n\",\n    \"from __future__ import annotations\\n\",\n    \"from typing import List, Dict, Any, Tuple\\n\",\n    \"import numpy as np, pandas as pd\\n\",\n    \"from sklearn.pipeline import Pipeline\\n\",\n    \"from sklearn.impute import SimpleImputer\\n\",\n    \"from sklearn.preprocessing import OneHotEncoder\\n\",\n    \"from sklearn.compose import ColumnTransformer\\n\",\n    \"from sklearn.linear_model import LogisticRegression\\n\",\n    \"from sklearn.calibration import CalibratedClassifierCV\\n\",\n    \"\\n\",\n    \"class TinyCritics:\\n\",\n    \"    def __init__(self):\\n\",\n    \"        base = Pipeline([(\\\"impute\\\", SimpleImputer(strategy=\\\"most_frequent\\\")),(\\\"clf\\\", LogisticRegression(max_iter=1000))])\\n\",\n    \"        self.model = CalibratedClassifierCV(base, method=\\\"isotonic\\\", cv=3)\\n\",\n    \"        self.num_features_: List[str] = [\\\"hour\\\",\\\"spo2\\\",\\\"backlog_ct\\\",\\\"backlog_lab\\\",\\\"backlog_ecg\\\",\\\"pending_n\\\",\\\"completed_n\\\",\\\"consults_n\\\",\\\"since_vitals_min\\\"]\\n\",\n    \"        self.cat_features_: List[str] = [\\\"role\\\",\\\"cp\\\",\\\"resp\\\",\\\"trauma\\\"]\\n\",\n    \"        self.preproc = ColumnTransformer([(\\\"num\\\", SimpleImputer(strategy=\\\"median\\\"), self.num_features_),(\\\"cat\\\", OneHotEncoder(handle_unknown=\\\"ignore\\\"), self.cat_features_)], remainder=\\\"drop\\\")\\n\",\n    \"        self.is_fit = False\\n\",\n    \"    def _featurize(self, X: List[Dict[str,Any]]) -> pd.DataFrame:\\n\",\n    \"        rows = []\\n\",\n    \"        for x in X:\\n\",\n    \"            s = x.get(\\\"state\\\"); a = x.get(\\\"action\\\", {})\\n\",\n    \"            if hasattr(s, \\\"feature_dict\\\"): f = s.feature_dict()\\n\",\n    \"            elif isinstance(s, dict): f = dict(s)\\n\",\n    \"            else: f = {}\\n\",\n    \"            f[\\\"action_label\\\"] = str(a.get(\\\"label\\\") or a.get(\\\"id\\\") or \\\"action\\\")\\n\",\n    \"            rows.append(f)\\n\",\n    \"        df = pd.DataFrame(rows)\\n\",\n    \"        for col in self.num_features_ + self.cat_features_:\\n\",\n    \"            if col not in df.columns: df[col] = np.nan if col in self.num_features_ else \\\"NA\\\"\\n\",\n    \"        return df[self.num_features_ + self.cat_features_ + [\\\"action_label\\\"]]\\n\",\n    \"    def fit(self, samples: List[Dict[str,Any]], y: np.ndarray) -> \\\"TinyCritics\\\":\\n\",\n    \"        df = self._featurize(samples)\\n\",\n    \"        Xp = self.preproc.fit_transform(df[self.num_features_ + self.cat_features_]); self.model.fit(Xp, y); self.is_fit = True; return self\\n\",\n    \"    def score(self, state, actions: List[Dict[str,Any]]):\\n\",\n    \"        X = self._featurize([{\\\"state\\\": state, \\\"action\\\": a} for a in actions])\\n\",\n    \"        if not self.is_fit:\\n\",\n    \"            n = len(actions); return np.full(n, 0.5), np.zeros(n), np.zeros(n)\\n\",\n    \"        Xp = self.preproc.transform(X[self.num_features_ + self.cat_features_])\\n\",\n    \"        p = self.model.predict_proba(Xp)[:, 1]\\n\",\n    \"        benefit = (1.0 - np.clip(X[\\\"backlog_ct\\\"].fillna(0), 0, 10)/10.0).to_numpy()\\n\",\n    \"        burden = (np.clip(X[\\\"since_vitals_min\\\"].fillna(60), 0, 120)/120.0).to_numpy()\\n\",\n    \"        return p, benefit, burden\\n\",\n    \"\\n\",\n    \"print(\\\"\u2705 TinyCritics ready\\\")\"\n   ]\n  },\n  {\n   \"cell_type\": \"code\",\n   \"execution_count\": null,\n   \"id\": \"workflow_extensions\",\n   \"metadata\": {},\n   \"outputs\": [],\n   \"source\": [\n    \"# V8 ENHANCEMENT: WORKFLOW STATE EXTENSIONS (CONTRACT COMPLIANT)\\n\",\n    \"\\n\",\n    \"def ensure_workflow_state_methods():\\n\",\n    \"    \\\"\\\"\\\"\\n\",\n    \"    Add required methods to WorkflowState without breaking existing functionality.\\n\",\n    \"    Contract-compliant: only extends, never removes or renames.\\n\",\n    \"    \\\"\\\"\\\"\\n\",\n    \"    \\n\",\n    \"    # Add feature_dict method if not present (required for TinyCritics)\\n\",\n    \"    if not hasattr(WorkflowState, 'feature_dict'):\\n\",\n    \"        def feature_dict(self):\\n\",\n    \"            \\\"\\\"\\\"Generate feature dictionary for TinyCritics compatibility.\\\"\\\"\\\"\\n\",\n    \"            # Base features for TinyCritics compatibility\\n\",\n    \"            features = {\\n\",\n    \"                \\\"hour\\\": self.hour,\\n\",\n    \"                \\\"spo2\\\": 98.0,  # Default value\\n\",\n    \"                \\\"backlog_ct\\\": self.backlog_ct,\\n\",\n    \"                \\\"backlog_lab\\\": self.backlog_lab,\\n\",\n    \"                \\\"backlog_ecg\\\": self.backlog_ecg,\\n\",\n    \"                \\\"pending_n\\\": len(self.pending_orders),\\n\",\n    \"                \\\"completed_n\\\": len(self.completed_studies),\\n\",\n    \"                \\\"consults_n\\\": len(self.active_consults),\\n\",\n    \"                \\\"since_vitals_min\\\": 0.0 if self.last_vitals_ts is None else \\n\",\n    \"                    (pd.Timestamp.utcnow() - self.last_vitals_ts).total_seconds() / 60.0,\\n\",\n    \"                \\\"role\\\": self.role,\\n\",\n    \"                \\\"cp\\\": int(self.chest_pain),\\n\",\n    \"                \\\"resp\\\": \\\"normal\\\",  # Default\\n\",\n    \"                \\\"trauma\\\": int(self.trauma)\\n\",\n    \"            }\\n\",\n    \"            \\n\",\n    \"            # V8: Operational features (always available)\\n\",\n    \"            features.update({\\n\",\n    \"                \\\"equipment_tracking_active\\\": True,\\n\",\n    \"                \\\"sop_access_available\\\": True,\\n\",\n    \"                \\\"lingering_check_enabled\\\": True\\n\",\n    \"            })\\n\",\n    \"            \\n\",\n    \"            # Phase 2 clinical extensions (only when enabled)\\n\",\n    \"            if RUN_PIPELINE:\\n\",\n    \"                features.update({\\n\",\n    \"                    \\\"troponin_pending\\\": int(\\\"LAB_TROPONIN\\\" in self.pending_orders),\\n\",\n    \"                    \\\"ecg_completed\\\": int(\\\"ORDER_ECG\\\" in self.completed_studies),\\n\",\n    \"                    \\\"ct_pending\\\": int(\\\"ORDER_CT\\\" in self.pending_orders),\\n\",\n    \"                    \\\"cardiology_consulted\\\": int(\\\"CARDIOLOGY\\\" in self.active_consults),\\n\",\n    \"                    \\\"clinical_deterioration\\\": int(\\\"Deteriorating\\\" in self.completed_studies),\\n\",\n    \"                    \\\"is_lingering\\\": features[\\\"since_vitals_min\\\"] > 120,  # >2 hours\\n\",\n    \"                    \\\"needs_reassessment\\\": features[\\\"since_vitals_min\\\"] > 240,  # >4 hours\\n\",\n    \"                })\\n\",\n    \"            \\n\",\n    \"            return features\\n\",\n    \"        \\n\",\n    \"        WorkflowState.feature_dict = feature_dict\\n\",\n    \"        print(\\\"\u2705 Added feature_dict method to WorkflowState\\\")\\n\",\n    \"    \\n\",\n    \"    # Add touch_now method if not present\\n\",\n    \"    if not hasattr(WorkflowState, 'touch_now'):\\n\",\n    \"        def touch_now(self, timestamp=None):\\n\",\n    \"            \\\"\\\"\\\"Update last vitals timestamp.\\\"\\\"\\\"\\n\",\n    \"            self.last_vitals_ts = timestamp or pd.Timestamp.utcnow()\\n\",\n    \"        \\n\",\n    \"        WorkflowState.touch_now = touch_now\\n\",\n    \"        print(\\\"\u2705 Added touch_now method to WorkflowState\\\")\\n\",\n    \"    \\n\",\n    \"    # Add lingering patient check method\\n\",\n    \"    if not hasattr(WorkflowState, 'is_lingering_patient'):\\n\",\n    \"        def is_lingering_patient(self, threshold_min: int = 120) -> bool:\\n\",\n    \"            \\\"\\\"\\\"Check if patient is lingering (overdue for assessment).\\\"\\\"\\\"\\n\",\n    \"            if self.last_vitals_ts is None:\\n\",\n    \"                return True  # No vitals recorded\\n\",\n    \"            \\n\",\n    \"            minutes_since = (pd.Timestamp.utcnow() - self.last_vitals_ts).total_seconds() / 60.0\\n\",\n    \"            return minutes_since > threshold_min\\n\",\n    \"        \\n\",\n    \"        WorkflowState.is_lingering_patient = is_lingering_patient\\n\",\n    \"        print(\\\"\u2705 Added is_lingering_patient method to WorkflowState\\\")\\n\",\n    \"\\n\",\n    \"# Initialize WorkflowState extensions\\n\",\n    \"ensure_workflow_state_methods()\\n\",\n    \"print(\\\"\u2705 Enhanced WorkflowState extensions ready\\\")\"\n   ]\n  },\n  {\n   \"cell_type\": \"code\",\n   \"execution_count\": null,\n   \"id\": \"troponin_rules\",\n   \"metadata\": {},\n   \"outputs\": [],\n   \"source\": [\n    \"# PHASE 1: CLINICAL RULES (CORRECTED TROPONIN LOGIC)\\n\",\n    \"def rule_hs_tnt(value):\\n\",\n    \"    \\\"\\\"\\\"\\n\",\n    \"    High-sensitivity troponin delta threshold calculator.\\n\",\n    \"    Clinical rule: <14 or >51 need 50% change, 15-50 need 20% change\\n\",\n    \"    \\\"\\\"\\\"\\n\",\n    \"    try: v = float(value)\\n\",\n    \"    except Exception: return 0.50  # Default to 50% if invalid\\n\",\n    \"    \\n\",\n    \"    if v < 14: return 0.50      # Below 14: need 50% change\\n\",\n    \"    if 15 <= v <= 50: return 0.20  # 15-50 range: need 20% change  \\n\",\n    \"    return 0.50                 # Above 51: need 50% change\\n\",\n    \"\\n\",\n    \"# Test the corrected logic\\n\",\n    \"assert rule_hs_tnt(13.9) == 0.50  # Below 14 -> 50%\\n\",\n    \"assert rule_hs_tnt(25.0) == 0.20  # 15-50 range -> 20%\\n\",\n    \"assert rule_hs_tnt(51.1) == 0.50  # Above 51 -> 50%\\n\",\n    \"print(\\\"\u2705 Corrected troponin delta rules ready\\\")\"\n   ]\n  },\n  {\n   \"cell_type\": \"code\",\n   \"execution_count\": null,\n   \"id\": \"equipment_tracking\",\n   \"metadata\": {},\n   \"outputs\": [],\n   \"source\": [\n    \"# PHASE 1: CORE EQUIPMENT TRACKING SYSTEM (PRESERVED FROM v6)\\n\",\n    \"from dataclasses import dataclass\\n\",\n    \"from typing import Any, Dict, List, Optional\\n\",\n    \"from pathlib import Path\\n\",\n    \"import pandas as pd, numpy as np\\n\",\n    \"\\n\",\n    \"def _cfg(CONFIG: Any, key: str, default: Any=None) -> Any:\\n\",\n    \"    try: return CONFIG.get(key, default)\\n\",\n    \"    except Exception: return getattr(CONFIG, key, default) if hasattr(CONFIG, key) else default\\n\",\n    \"\\n\",\n    \"def _ensure_parent(p: Path): p = Path(p); p.parent.mkdir(parents=True, exist_ok=True)\\n\",\n    \"\\n\",\n    \"@dataclass\\n\",\n    \"class EquipmentRecord:\\n\",\n    \"    equip_id: str; name: str=\\\"\\\"; location: str=\\\"\\\"; status: str=\\\"\\\"; last_seen: Optional[str]=None; battery: Optional[float]=None; confidence: Optional[float]=None\\n\",\n    \"    def to_row(self)->Dict[str,Any]: return {\\\"equip_id\\\":self.equip_id,\\\"name\\\":self.name,\\\"location\\\":self.location,\\\"status\\\":self.status,\\\"last_seen\\\":self.last_seen,\\\"battery\\\":self.battery,\\\"confidence\\\":self.confidence}\\n\",\n    \"\\n\",\n    \"class EquipmentRepository:\\n\",\n    \"    def __init__(self, status_csv: Path):\\n\",\n    \"        self.status_csv=Path(status_csv); _ensure_parent(self.status_csv)\\n\",\n    \"        if not self.status_csv.exists(): pd.DataFrame(columns=[\\\"equip_id\\\",\\\"name\\\",\\\"location\\\",\\\"status\\\",\\\"last_seen\\\",\\\"battery\\\",\\\"confidence\\\"]).to_csv(self.status_csv, index=False)\\n\",\n    \"    def read(self)->pd.DataFrame:\\n\",\n    \"        try: df=pd.read_csv(self.status_csv); \\n\",\n    \"        except Exception: return pd.DataFrame(columns=[\\\"equip_id\\\",\\\"name\\\",\\\"location\\\",\\\"status\\\",\\\"last_seen\\\",\\\"battery\\\",\\\"confidence\\\"])\\n\",\n    \"        if \\\"equip_id\\\" in df.columns: df[\\\"equip_id\\\"]=df[\\\"equip_id\\\"].astype(str); return df\\n\",\n    \"    def upsert(self, rec: EquipmentRecord)->None:\\n\",\n    \"        df=self.read(); row=pd.DataFrame([rec.to_row()])\\n\",\n    \"        if df.empty: df=row\\n\",\n    \"        else:\\n\",\n    \"            mask=(df[\\\"equip_id\\\"].astype(str)==str(rec.equip_id))\\n\",\n    \"            if mask.any(): df.loc[mask,:]=row.values\\n\",\n    \"            else: df=pd.concat([df,row], ignore_index=True)\\n\",\n    \"        df.to_csv(self.status_csv, index=False)\\n\",\n    \"\\n\",\n    \"class MovesLogRepository:\\n\",\n    \"    def __init__(self, moves_csv: Path):\\n\",\n    \"        self.moves_csv=Path(moves_csv); _ensure_parent(self.moves_csv)\\n\",\n    \"        if not self.moves_csv.exists(): pd.DataFrame(columns=[\\\"equip_id\\\",\\\"from\\\",\\\"to\\\",\\\"ts\\\"]).to_csv(self.moves_csv, index=False)\\n\",\n    \"    def append(self, equip_id:str, loc_from:str, loc_to:str, ts_iso:str)->None:\\n\",\n    \"        row=pd.DataFrame([{\\\"equip_id\\\":equip_id,\\\"from\\\":loc_from,\\\"to\\\":loc_to,\\\"ts\\\":ts_iso}])\\n\",\n    \"        try: prev=pd.read_csv(self.moves_csv) if self.moves_csv.exists() else None; df=pd.concat([prev,row], ignore_index=True) if prev is not None else row\\n\",\n    \"        except Exception: df=row\\n\",\n    \"        df.to_csv(self.moves_csv, index=False)\\n\",\n    \"    def read(self)->pd.DataFrame:\\n\",\n    \"        try: return pd.read_csv(self.moves_csv)\\n\",\n    \"        except Exception: return pd.DataFrame(columns=[\\\"equip_id\\\",\\\"from\\\",\\\"to\\\",\\\"ts\\\"])\\n\",\n    \"\\n\",\n    \"class SOPRegistry:\\n\",\n    \"    def __init__(self, sop_csv: Path): self.sop_csv=Path(sop_csv); _ensure_parent(self.sop_csv)\\n\",\n    \"    def read(self)->pd.DataFrame:\\n\",\n    \"        if self.sop_csv.exists():\\n\",\n    \"            try:\\n\",\n    \"                df=pd.read_csv(self.sop_csv)\\n\",\n    \"                for col in [\\\"sop_id\\\",\\\"title\\\",\\\"pdf_path\\\"]:\\n\",\n    \"                    if col not in df.columns: df[col]=\\\"\\\"\\n\",\n    \"                return df\\n\",\n    \"            except Exception: pass\\n\",\n    \"        return pd.DataFrame(columns=[\\\"sop_id\\\",\\\"title\\\",\\\"pdf_path\\\",\\\"version\\\",\\\"status\\\",\\\"keywords\\\",\\\"checklist\\\",\\\"source_url\\\"])\\n\",\n    \"\\n\",\n    \"class QRService:\\n\",\n    \"    def __init__(self,out_dir:Path): \\n\",\n    \"        self.out_dir=Path(out_dir); self.out_dir.mkdir(parents=True, exist_ok=True)\\n\",\n    \"    def make(self,payload:str)->str:\\n\",\n    \"        try:\\n\",\n    \"            import qrcode\\n\",\n    \"            fp=self.out_dir/f\\\"qr_{abs(hash(payload))}.png\\\"\\n\",\n    \"            img=qrcode.make(payload); img.save(fp); return str(fp)\\n\",\n    \"        except Exception: return f\\\"[QR fallback] {payload}\\\"\\n\",\n    \"    def decode_file(self, image_bytes:bytes):\\n\",\n    \"        try:\\n\",\n    \"            from PIL import Image; import io\\n\",\n    \"            img=Image.open(io.BytesIO(image_bytes))\\n\",\n    \"            try:\\n\",\n    \"                from pyzbar.pyzbar import decode as zbar_decode\\n\",\n    \"                res=zbar_decode(img); \\n\",\n    \"                if res: return res[0].data.decode(\\\"utf-8\\\",\\\"ignore\\\")\\n\",\n    \"            except Exception: pass\\n\",\n    \"        except Exception: pass\\n\",\n    \"        return None\\n\",\n    \"\\n\",\n    \"class TrackerService:\\n\",\n    \"    def __init__(self, equipment_repo:EquipmentRepository, moves_repo:MovesLogRepository, sop_registry:SOPRegistry, qr:QRService, config:Any):\\n\",\n    \"        self.equipment_repo=equipment_repo; self.moves_repo=moves_repo; self.sop_registry=sop_registry; self.qr=qr; self.CONFIG=config\\n\",\n    \"    @classmethod\\n\",\n    \"    def from_config(cls, CONFIG:Any)->\\\"TrackerService\\\":\\n\",\n    \"        return cls(EquipmentRepository(Path(_cfg(CONFIG,\\\"EQUIPMENT_STATUS_PATH\\\"))),\\n\",\n    \"                   MovesLogRepository(Path(_cfg(CONFIG,\\\"EQUIPMENT_MOVES_LOG_PATH\\\"))),\\n\",\n    \"                   SOPRegistry(Path(_cfg(CONFIG,\\\"SOP_REGISTRY_PATH\\\"))),\\n\",\n    \"                   QRService(Path(_cfg(CONFIG,\\\"QR_OUTPUT_DIR\\\"))), CONFIG)\\n\",\n    \"    def equipment_status(self)->pd.DataFrame: return self.equipment_repo.read()\\n\",\n    \"    def log_move(self, equip_id:str, loc_from:str, loc_to:str)->None:\\n\",\n    \"        ts_iso=pd.Timestamp.utcnow().isoformat(); df=self.equipment_repo.read()\\n\",\n    \"        row=df[df[\\\"equip_id\\\"].astype(str)==str(equip_id)]; name=row[\\\"name\\\"].iloc[0] if not row.empty and \\\"name\\\" in row.columns else \\\"\\\"\\n\",\n    \"        rec=EquipmentRecord(equip_id=equip_id,name=name,location=loc_to,status=\\\"moved\\\",last_seen=ts_iso)\\n\",\n    \"        self.equipment_repo.upsert(rec); self.moves_repo.append(equip_id, loc_from or \\\"\\\", loc_to, ts_iso)\\n\",\n    \"    def find_equipment(self, query:str)->pd.DataFrame:\\n\",\n    \"        q=(query or \\\"\\\").strip().lower(); df=self.equipment_repo.read()\\n\",\n    \"        if not q: return df\\n\",\n    \"        def hit(r): return any(q in str(r.get(k,\\\"\\\")).lower() for k in [\\\"equip_id\\\",\\\"name\\\",\\\"location\\\",\\\"status\\\"])\\n\",\n    \"        return df[df.apply(hit, axis=1)]\\n\",\n    \"    def overdue_equipment(self, threshold_minutes:int=120)->pd.DataFrame:\\n\",\n    \"        df=self.equipment_repo.read().copy()\\n\",\n    \"        if df.empty or \\\"last_seen\\\" not in df.columns: return df.iloc[0:0]\\n\",\n    \"        ts=pd.to_datetime(df[\\\"last_seen\\\"],errors=\\\"coerce\\\",utc=True); age_min=(pd.Timestamp.utcnow().tz_localize(\\\"UTC\\\")-ts).dt.total_seconds()/60.0\\n\",\n    \"        df[\\\"age_min\\\"]=age_min; return df[age_min>float(threshold_minutes)].sort_values(\\\"age_min\\\", ascending=False)\\n\",\n    \"    def movement_stats(self)->Dict[str,pd.DataFrame]:\\n\",\n    \"        log=self.moves_repo.read()\\n\",\n    \"        if log.empty: return {\\\"moves_per_equipment\\\":log,\\\"routes\\\":log}\\n\",\n    \"        per_eq=log.groupby(\\\"equip_id\\\").size().reset_index(name=\\\"moves\\\").sort_values(\\\"moves\\\", ascending=False)\\n\",\n    \"        routes=log.groupby([\\\"from\\\",\\\"to\\\"]).size().reset_index(name=\\\"count\\\").sort_values(\\\"count\\\", ascending=False)\\n\",\n    \"        return {\\\"moves_per_equipment\\\":per_eq,\\\"routes\\\":routes}\\n\",\n    \"    def sop_table(self)->pd.DataFrame: return self.sop_registry.read()\\n\",\n    \"    def search_sop(self, query:str)->pd.DataFrame:\\n\",\n    \"        df=self.sop_registry.read().copy(); q=(query or \\\"\\\").strip().lower()\\n\",\n    \"        if df.empty or not q: return df\\n\",\n    \"        cols=[c for c in [\\\"sop_id\\\",\\\"title\\\",\\\"keywords\\\",\\\"version\\\",\\\"status\\\"] if c in df.columns]\\n\",\n    \"        mask=df[cols].astype(str).apply(lambda col: col.str.lower().str.contains(q, na=False)).any(axis=1)\\n\",\n    \"        return df[mask]\\n\",\n    \"    def make_qr(self,payload:str)->str: return self.qr.make(payload)\\n\",\n    \"    def decode_qr_bytes(self, image_bytes:bytes): return self.qr.decode_file(image_bytes)\\n\",\n    \"\\n\",\n    \"print(\\\"\u2705 Core equipment tracking system ready (Phase 1 priority)\\\")\"\n   ]\n  },\n  {\n   \"cell_type\": \"code\",\n   \"execution_count\": null,\n   \"id\": \"sop_system\",\n   \"metadata\": {},\n   \"outputs\": [],\n   \"source\": [\n    \"# PHASE 1: SOP AUTO-PULL SYSTEM (PRESERVED FROM v6)\\n\",\n    \"from pathlib import Path\\n\",\n    \"from typing import Any, Dict, List\\n\",\n    \"\\n\",\n    \"def _slugify(text:str)->str:\\n\",\n    \"    import re; s=re.sub(r\\\"[^a-zA-Z0-9]+\\\",\\\"-\\\",text.strip().lower()).strip(\\\"-\\\"); return s or \\\"sop\\\"\\n\",\n    \"\\n\",\n    \"def refresh_sop_registry(CONFIG: Any, base_url: str=\\\"https://sop-notaufnahme.de/sop/\\\")->Dict[str,Any]:\\n\",\n    \"    out_csv=Path(CONFIG[\\\"SOP_REGISTRY_PATH\\\"]); pdf_dir=Path(CONFIG[\\\"DATA_ROOT\\\"])/\\\"sop_pdfs\\\"; pdf_dir.mkdir(parents=True, exist_ok=True)\\n\",\n    \"    try:\\n\",\n    \"        import requests; from bs4 import BeautifulSoup\\n\",\n    \"    except Exception as e:\\n\",\n    \"        return {\\\"found\\\":0,\\\"saved\\\":0,\\\"errors\\\":1,\\\"error\\\":f\\\"missing libs: {e}\\\"}\\n\",\n    \"    found=saved=errors=0; items=[]\\n\",\n    \"    try:\\n\",\n    \"        r=requests.get(base_url, timeout=15); r.raise_for_status(); soup=BeautifulSoup(r.text,\\\"html.parser\\\")\\n\",\n    \"        links=sorted({a[\\\"href\\\"] for a in soup.find_all(\\\"a\\\", href=True) if \\\"/product/\\\" in a[\\\"href\\\"] and a[\\\"href\\\"].startswith(\\\"http\\\")})\\n\",\n    \"        for url in links:\\n\",\n    \"            try:\\n\",\n    \"                pr=requests.get(url, timeout=15); pr.raise_for_status(); ps=BeautifulSoup(pr.text,\\\"html.parser\\\")\\n\",\n    \"                ttag=ps.find([\\\"h1\\\",\\\"h2\\\"]); title=ttag.get_text(strip=True) if ttag else (ps.find(\\\"title\\\").get_text(strip=True) if ps.find(\\\"title\\\") else url)\\n\",\n    \"                pdfs=[a[\\\"href\\\"] for a in ps.find_all(\\\"a\\\", href=True) if a[\\\"href\\\"].lower().endswith(\\\".pdf\\\")]\\n\",\n    \"                pdf_url=pdfs[0] if pdfs else None; sop_id=_slugify(title or url.split(\\\"/\\\")[-2]); pdf_path=\\\"\\\"\\n\",\n    \"                if pdf_url:\\n\",\n    \"                    try:\\n\",\n    \"                        fn=sop_id+\\\".pdf\\\"; outp=pdf_dir/fn\\n\",\n    \"                        with requests.get(pdf_url, stream=True, timeout=30) as dr:\\n\",\n    \"                            dr.raise_for_status()\\n\",\n    \"                            with open(outp,\\\"wb\\\") as f:\\n\",\n    \"                                for chunk in dr.iter_content(8192):\\n\",\n    \"                                    if chunk: f.write(chunk)\\n\",\n    \"                        pdf_path=str(outp); saved+=1\\n\",\n    \"                    except Exception:\\n\",\n    \"                        errors+=1; pdf_path=pdf_url\\n\",\n    \"                items.append({\\\"sop_id\\\":sop_id,\\\"title\\\":title or sop_id,\\\"pdf_path\\\":pdf_path,\\\"version\\\":\\\"\\\",\\\"status\\\":\\\"fetched\\\" if pdf_path else \\\"linked\\\",\\\"keywords\\\":\\\"\\\",\\\"checklist\\\":\\\"\\\",\\\"source_url\\\":url})\\n\",\n    \"                found+=1\\n\",\n    \"            except Exception: errors+=1; continue\\n\",\n    \"    except Exception as e:\\n\",\n    \"        return {\\\"found\\\":0,\\\"saved\\\":0,\\\"errors\\\":1,\\\"error\\\":str(e)}\\n\",\n    \"    import pandas as pd\\n\",\n    \"    try:\\n\",\n    \"        if out_csv.exists(): df=pd.read_csv(out_csv)\\n\",\n    \"        else: df=pd.DataFrame(columns=[\\\"sop_id\\\",\\\"title\\\",\\\"pdf_path\\\",\\\"version\\\",\\\"status\\\",\\\"keywords\\\",\\\"checklist\\\",\\\"source_url\\\"])\\n\",\n    \"        df=df.copy()\\n\",\n    \"        if df.empty: new_df=pd.DataFrame(items)\\n\",\n    \"        else:\\n\",\n    \"            df[\\\"sop_id\\\"]=df[\\\"sop_id\\\"].astype(str)\\n\",\n    \"            for i in items:\\n\",\n    \"                mask=(df[\\\"sop_id\\\"]==str(i[\\\"sop_id\\\"]))\\n\",\n    \"                if mask.any():\\n\",\n    \"                    for k,v in i.items():\\n\",\n    \"                        if k in df.columns and (pd.isna(df.loc[mask,k]).all() or str(df.loc[mask,k].iloc[0]).strip()==\\\"\\\" or k in [\\\"pdf_path\\\",\\\"status\\\",\\\"source_url\\\"]):\\n\",\n    \"                            df.loc[mask,k]=v\\n\",\n    \"                else:\\n\",\n    \"                    df=pd.concat([df, pd.DataFrame([i])], ignore_index=True)\\n\",\n    \"            new_df=df\\n\",\n    \"        new_df.to_csv(out_csv, index=False)\\n\",\n    \"    except Exception as e:\\n\",\n    \"        errors+=1\\n\",\n    \"    return {\\\"found\\\":found,\\\"saved\\\":saved,\\\"errors\\\":errors,\\\"csv\\\":str(out_csv),\\\"dir\\\":str(pdf_dir)}\\n\",\n    \"\\n\",\n    \"def load_priority_flows(json_path:str)->Dict[str,Any]:\\n\",\n    \"    import json\\n\",\n    \"    try:\\n\",\n    \"        with open(json_path,\\\"r\\\",encoding=\\\"utf-8\\\") as f: return json.load(f)\\n\",\n    \"    except Exception: return {}\\n\",\n    \"\\n\",\n    \"print(\\\"\u2705 SOP auto-pull system ready (Phase 1 priority)\\\")\"\n   ]\n  },\n  {\n   \"cell_type\": \"code\",\n   \"execution_count\": null,\n   \"id\": \"phase2_clinical\",\n   \"metadata\": {},\n   \"outputs\": [],\n   \"source\": [\n    \"# V8 ENHANCEMENT: PHASE 2 CLINICAL SYSTEMS (GUARDED BY RUN_PIPELINE)\\n\",\n    \"\\n\",\n    \"if RUN_PIPELINE:\\n\",\n    \"    print(\\\"\ud83d\udd2c Initializing Phase 2 clinical systems...\\\")\\n\",\n    \"    \\n\",\n    \"    # Enhanced clinical logic from v6 enhanced notebook\\n\",\n    \"    from datetime import datetime, timedelta\\n\",\n    \"    from typing import Callable, Dict, Any, List, Tuple, Optional\\n\",\n    \"    import pandas as pd\\n\",\n    \"    import re\\n\",\n    \"    \\n\",\n    \"    # Complete ResultsNotifier from enhanced v6\\n\",\n    \"    class ResultsNotifier:\\n\",\n    \"        def __init__(self):\\n\",\n    \"            self.callbacks: List[Callable[[str, str, Dict[str, Any]], None]] = []\\n\",\n    \"            self.last_values: Dict[str, Dict[str, Tuple[float, datetime]]] = {}\\n\",\n    \"            # Note: Troponin delta rules are value-dependent, not fixed thresholds\\n\",\n    \"        def on_notify(self, fn): self.callbacks.append(fn)\\n\",\n    \"        def _emit(self, pid, event, payload):\\n\",\n    \"            for cb in self.callbacks: cb(pid, event, payload)\\n\",\n    \"        @staticmethod\\n\",\n    \"        def _split_segments(msg): return [s.split(\\\"|\\\") for s in msg.strip().split(\\\"\\\\r\\\") if s]\\n\",\n    \"        @staticmethod\\n\",\n    \"        def _field(component, idx):\\n\",\n    \"            parts = component.split(\\\"^\\\"); return parts[idx] if idx < len(parts) else \\\"\\\"\\n\",\n    \"        @staticmethod\\n\",\n    \"        def _parse_ts(ts):\\n\",\n    \"            for fmt in (\\\"%Y%m%d%H%M%S\\\",\\\"%Y%m%d%H%M\\\",\\\"%Y%m%d\\\"):\\n\",\n    \"                try: return datetime.strptime(ts, fmt)\\n\",\n    \"                except: pass\\n\",\n    \"            return None\\n\",\n    \"        def _get_pid(self, segs):\\n\",\n    \"            for s in segs:\\n\",\n    \"                if s[0]==\\\"PID\\\": return s[3].split(\\\"^\\\")[0] if len(s)>3 else \\\"\\\"\\n\",\n    \"            return \\\"\\\"\\n\",\n    \"        def _get_troponin_delta_threshold(self, baseline_value: float) -> float:\\n\",\n    \"            \\\"\\\"\\\"\\n\",\n    \"            Get troponin delta threshold based on baseline value.\\n\",\n    \"            Clinical rule: <14 or >51 need 50%, 15-50 need 20%\\n\",\n    \"            \\\"\\\"\\\"\\n\",\n    \"            if baseline_value < 14:\\n\",\n    \"                return 0.50  # 50%\\n\",\n    \"            elif 15 <= baseline_value <= 50:\\n\",\n    \"                return 0.20  # 20% \\n\",\n    \"            else:  # > 51\\n\",\n    \"                return 0.50  # 50%\\n\",\n    \"        \\n\",\n    \"        def handle_hl7(self, message: str):\\n\",\n    \"            segs = self._split_segments(message)\\n\",\n    \"            if not segs or segs[0][0]!=\\\"MSH\\\": return\\n\",\n    \"            msg_type = segs[0][8] if len(segs[0])>8 else \\\"\\\"\\n\",\n    \"            pid = self._get_pid(segs) or \\\"UNKNOWN\\\"\\n\",\n    \"            if \\\"ORU^R01\\\" in msg_type: self._handle_oru(pid, segs)\\n\",\n    \"            elif \\\"MDM^T02\\\" in msg_type or (\\\"ORU^R01\\\" in msg_type and any(s[0]==\\\"OBX\\\" and s[2] in (\\\"TX\\\",\\\"FT\\\",\\\"ED\\\") for s in segs)):\\n\",\n    \"                self._handle_report(pid, segs)\\n\",\n    \"        def _handle_oru(self, pid, segs):\\n\",\n    \"            obr_accession, obr_ts = None, None\\n\",\n    \"            for s in segs:\\n\",\n    \"                if s[0]==\\\"OBR\\\":\\n\",\n    \"                    obr_accession = s[3] if len(s)>3 else None\\n\",\n    \"                    obr_ts = self._parse_ts(s[7]) if len(s)>7 else None\\n\",\n    \"                if s[0]==\\\"OBX\\\":\\n\",\n    \"                    id_comp = s[3] if len(s)>3 else \\\"\\\"\\n\",\n    \"                    code = self._field(id_comp,0) or self._field(id_comp,1) or \\\"UNKNOWN_TEST\\\"\\n\",\n    \"                    value_raw = s[5] if len(s)>5 else \\\"\\\"\\n\",\n    \"                    units = s[6] if len(s)>6 else \\\"\\\"\\n\",\n    \"                    status = s[11] if len(s)>11 else \\\"\\\"\\n\",\n    \"                    ts = self._parse_ts(s[14]) if len(s)>14 else obr_ts\\n\",\n    \"                    try: value = float(value_raw)\\n\",\n    \"                    except: value = None\\n\",\n    \"                    payload = {\\\"test_code\\\":code.upper(),\\\"value_raw\\\":value_raw,\\\"value\\\":value,\\\"units\\\":units,\\\"status\\\":status,\\\"ts\\\":ts,\\\"accession\\\":obr_accession}\\n\",\n    \"                    self._emit(pid,\\\"lab_result_ready\\\",payload)\\n\",\n    \"                    if status and status.upper().startswith(\\\"C\\\"): self._emit(pid,\\\"lab_result_critical\\\",payload)\\n\",\n    \"                    if value is not None: self._maybe_emit_delta(pid, payload)\\n\",\n    \"        def _maybe_emit_delta(self, pid, payload):\\n\",\n    \"            code, value = payload[\\\"test_code\\\"], payload[\\\"value\\\"]\\n\",\n    \"            ts = payload[\\\"ts\\\"] or datetime.utcnow()\\n\",\n    \"            \\n\",\n    \"            # Special handling for troponin with value-dependent thresholds\\n\",\n    \"            if code == \\\"TROPONIN\\\":\\n\",\n    \"                last = self.last_values.get(pid,{}).get(code)\\n\",\n    \"                if last:\\n\",\n    \"                    prev, _ = last\\n\",\n    \"                    abs_delta = abs(value - prev)\\n\",\n    \"                    rel_pct = (abs_delta/prev*100.0) if prev else 0.0\\n\",\n    \"                    \\n\",\n    \"                    # Use value-dependent threshold for troponin\\n\",\n    \"                    threshold_pct = self._get_troponin_delta_threshold(prev) * 100.0\\n\",\n    \"                    \\n\",\n    \"                    if rel_pct >= threshold_pct:\\n\",\n    \"                        delta_payload = {\\n\",\n    \"                            **payload,\\n\",\n    \"                            \\\"prev_value\\\": prev,\\n\",\n    \"                            \\\"abs_delta\\\": abs_delta,\\n\",\n    \"                            \\\"rel_pct\\\": rel_pct,\\n\",\n    \"                            \\\"threshold_used\\\": threshold_pct,\\n\",\n    \"                            \\\"rule\\\": f\\\"Baseline {prev} ng/L -> {threshold_pct}% threshold\\\"\\n\",\n    \"                        }\\n\",\n    \"                        self._emit(pid,\\\"lab_delta_positive\\\",delta_payload)\\n\",\n    \"                self.last_values.setdefault(pid,{})[code] = (value, ts)\\n\",\n    \"            else:\\n\",\n    \"                # For non-troponin tests, store the value but no delta logic yet\\n\",\n    \"                self.last_values.setdefault(pid,{})[code] = (value, ts)\\n\",\n    \"        def _handle_report(self, pid, segs):\\n\",\n    \"            text_blocks, study_id, ts = [], None, None\\n\",\n    \"            for s in segs:\\n\",\n    \"                if s[0]==\\\"OBR\\\":\\n\",\n    \"                    study_id = s[3] if len(s)>3 else study_id\\n\",\n    \"                    ts = self._parse_ts(s[7]) if len(s)>7 else ts\\n\",\n    \"                if s[0]==\\\"OBX\\\" and len(s)>2 and s[2] in (\\\"TX\\\",\\\"FT\\\",\\\"ED\\\"):\\n\",\n    \"                    text_blocks.append(s[5] if len(s)>5 else \\\"\\\")\\n\",\n    \"            if text_blocks:\\n\",\n    \"                self._emit(pid,\\\"imaging_report_ready\\\",{\\\"study_id\\\":study_id,\\\"report_text\\\":\\\"\\\\n\\\".join(text_blocks),\\\"ts\\\":ts})\\n\",\n    \"    \\n\",\n    \"    # Initialize Phase 2 components\\n\",\n    \"    RESULTS_NOTIFIER = ResultsNotifier()\\n\",\n    \"    \\n\",\n    \"    # Basic callback for demo\\n\",\n    \"    RESULTS_NOTIFIER.on_notify(\\n\",\n    \"        lambda pid, ev, payload: print(f\\\"\ud83d\udd2c [CLINICAL] {pid} - {ev} - {payload.get('test_code', payload.get('study_id', ''))}\\\")\\n\",\n    \"    )\\n\",\n    \"    \\n\",\n    \"    print(\\\"\u2705 Phase 2 clinical systems initialized\\\")\\n\",\n    \"    \\n\",\n    \"else:\\n\",\n    \"    print(\\\"\u23f8\ufe0f  Phase 2 clinical systems disabled (RUN_PIPELINE=False)\\\")\\n\",\n    \"    RESULTS_NOTIFIER = None\"\n   ]\n  },\n  {\n   \"cell_type\": \"code\",\n   \"execution_count\": null,\n   \"id\": \"lingering_monitor\",\n   \"metadata\": {},\n   \"outputs\": [],\n   \"source\": [\n    \"# V8 ENHANCEMENT: ENHANCED LINGERING PATIENT MONITORING\\n\",\n    \"\\n\",\n    \"class LingeringPatientMonitor:\\n\",\n    \"    \\\"\\\"\\\"\\n\",\n    \"    Enhanced lingering patient monitor - Phase 1 operational priority.\\n\",\n    \"    Source: Clinical Requirements - \\\"Stable patients linger in ED due to overcrowding\\\"\\n\",\n    \"    \\\"\\\"\\\"\\n\",\n    \"    \\n\",\n    \"    def __init__(self):\\n\",\n    \"        self.patients: Dict[str, Dict[str, Any]] = {}\\n\",\n    \"        self.alert_thresholds = {\\n\",\n    \"            \\\"assessment_overdue_min\\\": 120,  # >2 hours without assessment\\n\",\n    \"            \\\"vitals_overdue_min\\\": 240,      # >4 hours without vitals\\n\",\n    \"            \\\"basic_needs_min\\\": 360,         # >6 hours without food/comfort\\n\",\n    \"        }\\n\",\n    \"    \\n\",\n    \"    def register_patient(self, patient_id: str, workflow_state: WorkflowState):\\n\",\n    \"        \\\"\\\"\\\"Register patient for lingering monitoring.\\\"\\\"\\\"\\n\",\n    \"        now = pd.Timestamp.utcnow()\\n\",\n    \"        self.patients[patient_id] = {\\n\",\n    \"            \\\"workflow_state\\\": workflow_state,\\n\",\n    \"            \\\"registered_at\\\": now,\\n\",\n    \"            \\\"last_check\\\": now,\\n\",\n    \"            \\\"red_flags\\\": []\\n\",\n    \"        }\\n\",\n    \"    \\n\",\n    \"    def update_patient(self, patient_id: str, workflow_state: WorkflowState):\\n\",\n    \"        \\\"\\\"\\\"Update patient's workflow state.\\\"\\\"\\\"\\n\",\n    \"        if patient_id in self.patients:\\n\",\n    \"            self.patients[patient_id][\\\"workflow_state\\\"] = workflow_state\\n\",\n    \"            self.patients[patient_id][\\\"last_check\\\"] = pd.Timestamp.utcnow()\\n\",\n    \"    \\n\",\n    \"    def check_lingering_patients(self) -> List[Dict[str, Any]]:\\n\",\n    \"        \\\"\\\"\\\"Check for patients who are lingering and need attention.\\\"\\\"\\\"\\n\",\n    \"        alerts = []\\n\",\n    \"        \\n\",\n    \"        for patient_id, patient_data in self.patients.items():\\n\",\n    \"            state = patient_data[\\\"workflow_state\\\"]\\n\",\n    \"            \\n\",\n    \"            # Use the WorkflowState's feature_dict to get current status\\n\",\n    \"            if hasattr(state, 'feature_dict'):\\n\",\n    \"                features = state.feature_dict()\\n\",\n    \"                since_vitals = features.get(\\\"since_vitals_min\\\", 0)\\n\",\n    \"                \\n\",\n    \"                # Check if patient is lingering\\n\",\n    \"                if since_vitals > self.alert_thresholds[\\\"assessment_overdue_min\\\"]:\\n\",\n    \"                    alert = {\\n\",\n    \"                        \\\"patient_id\\\": patient_id,\\n\",\n    \"                        \\\"type\\\": \\\"lingering_patient\\\",\\n\",\n    \"                        \\\"severity\\\": self._determine_severity(since_vitals),\\n\",\n    \"                        \\\"since_vitals_min\\\": since_vitals,\\n\",\n    \"                        \\\"recommended_actions\\\": self._get_recommendations(since_vitals, features),\\n\",\n    \"                        \\\"timestamp\\\": pd.Timestamp.utcnow().isoformat()\\n\",\n    \"                    }\\n\",\n    \"                    alerts.append(alert)\\n\",\n    \"                    \\n\",\n    \"                    # Update red flags\\n\",\n    \"                    patient_data[\\\"red_flags\\\"].append({\\n\",\n    \"                        \\\"type\\\": \\\"lingering_detected\\\",\\n\",\n    \"                        \\\"timestamp\\\": pd.Timestamp.utcnow(),\\n\",\n    \"                        \\\"since_vitals\\\": since_vitals\\n\",\n    \"                    })\\n\",\n    \"        \\n\",\n    \"        return alerts\\n\",\n    \"    \\n\",\n    \"    def _determine_severity(self, since_vitals_min: float) -> str:\\n\",\n    \"        \\\"\\\"\\\"Determine severity based on time since last vitals.\\\"\\\"\\\"\\n\",\n    \"        if since_vitals_min > self.alert_thresholds[\\\"basic_needs_min\\\"]:\\n\",\n    \"            return \\\"critical\\\"  # >6 hours\\n\",\n    \"        elif since_vitals_min > self.alert_thresholds[\\\"vitals_overdue_min\\\"]:\\n\",\n    \"            return \\\"high\\\"     # >4 hours\\n\",\n    \"        elif since_vitals_min > self.alert_thresholds[\\\"assessment_overdue_min\\\"]:\\n\",\n    \"            return \\\"medium\\\"   # >2 hours\\n\",\n    \"        else:\\n\",\n    \"            return \\\"low\\\"\\n\",\n    \"    \\n\",\n    \"    def _get_recommendations(self, since_vitals_min: float, features: Dict) -> List[str]:\\n\",\n    \"        \\\"\\\"\\\"Get recommendations based on patient status.\\\"\\\"\\\"\\n\",\n    \"        recommendations = []\\n\",\n    \"        \\n\",\n    \"        if since_vitals_min > self.alert_thresholds[\\\"basic_needs_min\\\"]:\\n\",\n    \"            recommendations.extend([\\n\",\n    \"                \\\"IMMEDIATE_PHYSICIAN_REVIEW\\\",\\n\",\n    \"                \\\"CHECK_BASIC_NEEDS\\\",\\n\",\n    \"                \\\"CONSIDER_DISCHARGE_READINESS\\\",\\n\",\n    \"                \\\"SOCIAL_WORK_CONSULT\\\"\\n\",\n    \"            ])\\n\",\n    \"        elif since_vitals_min > self.alert_thresholds[\\\"vitals_overdue_min\\\"]:\\n\",\n    \"            recommendations.extend([\\n\",\n    \"                \\\"NURSING_ASSESSMENT\\\",\\n\",\n    \"                \\\"VITAL_SIGNS_OVERDUE\\\", \\n\",\n    \"                \\\"CHECK_PENDING_RESULTS\\\"\\n\",\n    \"            ])\\n\",\n    \"        elif since_vitals_min > self.alert_thresholds[\\\"assessment_overdue_min\\\"]:\\n\",\n    \"            recommendations.extend([\\n\",\n    \"                \\\"ROUTINE_VITALS_DUE\\\",\\n\",\n    \"                \\\"COMFORT_ROUNDS\\\"\\n\",\n    \"            ])\\n\",\n    \"        \\n\",\n    \"        # Add specific recommendations based on workflow state\\n\",\n    \"        if features.get(\\\"troponin_pending\\\"):\\n\",\n    \"            recommendations.append(\\\"FOLLOW_UP_TROPONIN_RESULTS\\\")\\n\",\n    \"        \\n\",\n    \"        if features.get(\\\"ct_pending\\\"):\\n\",\n    \"            recommendations.append(\\\"CHECK_IMAGING_DELAYS\\\")\\n\",\n    \"        \\n\",\n    \"        if features.get(\\\"cardiology_consulted\\\") and since_vitals_min > 180:\\n\",\n    \"            recommendations.append(\\\"FOLLOW_UP_CARDIOLOGY_RECOMMENDATIONS\\\")\\n\",\n    \"        \\n\",\n    \"        return recommendations\\n\",\n    \"    \\n\",\n    \"    def get_summary_stats(self) -> Dict[str, Any]:\\n\",\n    \"        \\\"\\\"\\\"Get summary statistics for lingering patients.\\\"\\\"\\\"\\n\",\n    \"        total = len(self.patients)\\n\",\n    \"        lingering = 0\\n\",\n    \"        overdue = 0\\n\",\n    \"        \\n\",\n    \"        for patient_data in self.patients.values():\\n\",\n    \"            state = patient_data[\\\"workflow_state\\\"]\\n\",\n    \"            if hasattr(state, 'is_lingering_patient'):\\n\",\n    \"                if state.is_lingering_patient(120):  # 2 hours\\n\",\n    \"                    lingering += 1\\n\",\n    \"                if state.is_lingering_patient(240):  # 4 hours\\n\",\n    \"                    overdue += 1\\n\",\n    \"        \\n\",\n    \"        return {\\n\",\n    \"            \\\"total_patients\\\": total,\\n\",\n    \"            \\\"lingering_patients\\\": lingering,\\n\",\n    \"            \\\"overdue_patients\\\": overdue,\\n\",\n    \"            \\\"percentage_lingering\\\": (lingering / total * 100.0) if total > 0 else 0.0\\n\",\n    \"        }\\n\",\n    \"\\n\",\n    \"# Initialize lingering monitor\\n\",\n    \"LINGERING_MONITOR = LingeringPatientMonitor()\\n\",\n    \"print(\\\"\u2705 Enhanced lingering patient monitoring ready (Phase 1 priority)\\\")\"\n   ]\n  },\n  {\n   \"cell_type\": \"code\",\n   \"execution_count\": null,\n   \"id\": \"phase2_bridge\",\n   \"metadata\": {},\n   \"outputs\": [],\n   \"source\": [\n    \"# PHASE 2 BRIDGE (PRESERVED FROM v6)\\n\",\n    \"import importlib\\n\",\n    \"def _try_import(name:str):\\n\",\n    \"    try: return importlib.import_module(name)\\n\",\n    \"    except Exception: return None\\n\",\n    \"def run_icu_constraints(state):\\n\",\n    \"    if not RUN_PIPELINE: return {}\\n\",\n    \"    mod = _try_import(\\\"icu_constraints\\\") or _try_import(\\\"modeling_icu_constraints\\\")\\n\",\n    \"    if mod and hasattr(mod,\\\"compute_icu_flags\\\"):\\n\",\n    \"        try: return dict(mod.compute_icu_flags(state))\\n\",\n    \"        except Exception: return {}\\n\",\n    \"    return {}\\n\",\n    \"def mesh_route_actions(state, actions):\\n\",\n    \"    if not RUN_PIPELINE: return actions\\n\",\n    \"    mod = _try_import(\\\"agent_mesh\\\") or _try_import(\\\"ed_agent_mesh\\\")\\n\",\n    \"    if mod and hasattr(mod,\\\"route\\\"):\\n\",\n    \"        try: return list(mod.route(state, actions))\\n\",\n    \"        except Exception: return actions\\n\",\n    \"    return actions\\n\",\n    \"def trainer_fit_critic(critic, samples, y):\\n\",\n    \"    if not RUN_PIPELINE: return critic\\n\",\n    \"    mod = _try_import(\\\"trainer\\\") or _try_import(\\\"ed_trainer\\\")\\n\",\n    \"    if mod and hasattr(mod,\\\"fit_critic\\\"):\\n\",\n    \"        try: return mod.fit_critic(critic, samples, y)\\n\",\n    \"        except Exception: return critic\\n\",\n    \"    return critic\\n\",\n    \"print(\\\"\u2705 Phase 2 bridge ready\\\")\"\n   ]\n  },\n  {\n   \"cell_type\": \"code\",\n   \"execution_count\": null,\n   \"id\": \"streamlit_ui\",\n   \"metadata\": {},\n   \"outputs\": [],\n   \"source\": [\n    \"# PHASE 1: STREAMLIT UI (ENHANCED FOR v8)\\n\",\n    \"def run_ui(tracker, get_state, get_actions, critic):\\n\",\n    \"    import streamlit as st, pandas as pd, numpy as np\\n\",\n    \"    st.set_page_config(page_title=\\\"ED Pipeline v8 - Complete\\\", layout=\\\"wide\\\")\\n\",\n    \"    st.title(\\\"\ud83c\udfe5 ED Pipeline v8 - Operational + Clinical\\\")\\n\",\n    \"    \\n\",\n    \"    # V8: System status indicator\\n\",\n    \"    col_status1, col_status2, col_status3 = st.columns([1,1,1])\\n\",\n    \"    with col_status1:\\n\",\n    \"        st.metric(\\\"Phase 1 Status\\\", \\\"\u2705 Active\\\", \\\"Equipment + SOP + Lingering\\\")\\n\",\n    \"    with col_status2:\\n\",\n    \"        phase2_status = \\\"\u2705 Active\\\" if RUN_PIPELINE else \\\"\u23f8\ufe0f Disabled\\\"\\n\",\n    \"        st.metric(\\\"Phase 2 Status\\\", phase2_status, \\\"Clinical Systems\\\")\\n\",\n    \"    with col_status3:\\n\",\n    \"        lingering_stats = LINGERING_MONITOR.get_summary_stats()\\n\",\n    \"        st.metric(\\\"Lingering Patients\\\", f\\\"{lingering_stats['lingering_patients']}/{lingering_stats['total_patients']}\\\", \\\"Active Monitoring\\\")\\n\",\n    \"    \\n\",\n    \"    # Original UI from v6 baseline (Phase 1 priority)\\n\",\n    \"    c0, c1, c2, c3 = st.columns([2,2,2,2])\\n\",\n    \"    with c0:\\n\",\n    \"        thresh = st.number_input(\\\"Overdue threshold (min)\\\", min_value=5, max_value=720, value=120, step=5)\\n\",\n    \"    with c1:\\n\",\n    \"        if st.button(\\\"Refresh\\\"): st.experimental_rerun()\\n\",\n    \"    \\n\",\n    \"    # PHASE 1: Equipment Tracking (Priority)\\n\",\n    \"    st.header(\\\"\ud83d\udd27 Equipment Tracking\\\")\\n\",\n    \"    eq_df = tracker.equipment_status()\\n\",\n    \"    s1, s2 = st.columns([2,1])\\n\",\n    \"    with s1:\\n\",\n    \"        q = st.text_input(\\\"Find equipment (ID / name / location / status)\\\", \\\"\\\")\\n\",\n    \"        filt = tracker.find_equipment(q) if q else eq_df\\n\",\n    \"        st.dataframe(filt, use_container_width=True, height=260)\\n\",\n    \"    with s2:\\n\",\n    \"        overdue = tracker.overdue_equipment(int(thresh))\\n\",\n    \"        st.subheader(\\\"Overdue\\\")\\n\",\n    \"        if overdue.empty: st.write(\\\"None\\\")\\n\",\n    \"        else: st.dataframe(overdue[[\\\"equip_id\\\",\\\"name\\\",\\\"location\\\",\\\"last_seen\\\",\\\"age_min\\\"]], use_container_width=True, height=200)\\n\",\n    \"    \\n\",\n    \"    st.markdown(\\\"**Update location / log move**\\\")\\n\",\n    \"    mc1, mc2, mc3, mc4 = st.columns([2,2,2,1])\\n\",\n    \"    with mc1: sel_id = st.selectbox(\\\"Equipment ID\\\", [\\\"\\\"] + sorted(list(eq_df.get(\\\"equip_id\\\", []))))\\n\",\n    \"    with mc2: loc_from = st.text_input(\\\"From\\\", \\\"\\\")\\n\",\n    \"    with mc3: loc_to = st.text_input(\\\"To\\\", \\\"\\\")\\n\",\n    \"    with mc4:\\n\",\n    \"        if st.button(\\\"Log move\\\") and sel_id and loc_to:\\n\",\n    \"            tracker.log_move(sel_id, loc_from, loc_to); st.success(f\\\"Move logged: {sel_id} \u2192 {loc_to}\\\")\\n\",\n    \"    \\n\",\n    \"    # PHASE 1: QR Code System (Priority)\\n\",\n    \"    st.header(\\\"\ud83d\udcf1 QR Code System\\\")\\n\",\n    \"    qr_col1, qr_col2 = st.columns([2,2])\\n\",\n    \"    with qr_col1:\\n\",\n    \"        qr_txt = st.text_input(\\\"QR payload to generate\\\", \\\"\\\")\\n\",\n    \"        if st.button(\\\"Generate QR\\\") and qr_txt:\\n\",\n    \"            path = tracker.make_qr(qr_txt); st.write(\\\"QR saved to:\\\", path)\\n\",\n    \"    with qr_col2:\\n\",\n    \"        st.write(\\\"Scan and update location\\\")\\n\",\n    \"        f = st.file_uploader(\\\"Upload QR image\\\", type=[\\\"png\\\",\\\"jpg\\\",\\\"jpeg\\\",\\\"webp\\\"])\\n\",\n    \"        manual_payload = st.text_input(\\\"Manual payload (fallback if decoding fails)\\\", \\\"\\\")\\n\",\n    \"        new_loc = st.text_input(\\\"New location (after scan)\\\", \\\"\\\")\\n\",\n    \"        if st.button(\\\"Scan & Update\\\"):\\n\",\n    \"            equip_payload = None\\n\",\n    \"            if f is not None: equip_payload = tracker.decode_qr_bytes(f.read())\\n\",\n    \"            if not equip_payload and manual_payload: equip_payload = manual_payload\\n\",\n    \"            if equip_payload and new_loc:\\n\",\n    \"                equip_id = equip_payload\\n\",\n    \"                if \\\"id=\\\" in equip_payload:\\n\",\n    \"                    try: equip_id = equip_payload.split(\\\"id=\\\",1)[1].split(\\\"&\\\",1)[0]\\n\",\n    \"                    except Exception: equip_id = equip_payload\\n\",\n    \"                tracker.log_move(str(equip_id), \\\"\\\", new_loc); st.success(f\\\"Updated via payload. {equip_id} \u2192 {new_loc}\\\")\\n\",\n    \"            elif not new_loc: st.error(\\\"Provide a new location.\\\")\\n\",\n    \"            else: st.error(\\\"No QR payload detected (image or manual).\\\")\\n\",\n    \"    \\n\",\n    \"    # PHASE 1: SOP System (Priority)\\n\",\n    \"    with st.expander(\\\"\ud83d\udccb SOP Auto-Pull and Protocols\\\", expanded=False):\\n\",\n    \"        if st.button(\\\"Refresh SOPs from sop-notaufnahme.de\\\"):\\n\",\n    \"            res = refresh_sop_registry(CONFIG, base_url=\\\"https://sop-notaufnahme.de/sop/\\\"); st.write(res)\\n\",\n    \"        flows = load_priority_flows(\\\"/mnt/data/priority_flows.json\\\")\\n\",\n    \"        if flows:\\n\",\n    \"            keys = sorted(list(flows.keys())); pickf = st.selectbox(\\\"Show flow\\\", [\\\"\\\"] + keys)\\n\",\n    \"            if pickf:\\n\",\n    \"                flow = flows[pickf]; st.subheader(flow.get(\\\"title\\\", pickf))\\n\",\n    \"                nodes = flow.get(\\\"nodes\\\", []); edges = flow.get(\\\"edges\\\", [])\\n\",\n    \"                st.write(\\\"Nodes:\\\", \\\", \\\".join([n.get(\\\"label\\\", n.get(\\\"id\\\",\\\"\\\")) for n in nodes]))\\n\",\n    \"    \\n\",\n    \"    st.header(\\\"\ud83d\udccb SOPs\\\")\\n\",\n    \"    sop_q = st.text_input(\\\"Search SOPs (id/title/keywords)\\\", \\\"\\\")\\n\",\n    \"    sop_hits = tracker.search_sop(sop_q)\\n\",\n    \"    if sop_hits.empty: st.info(\\\"No SOPs found.\\\")\\n\",\n    \"    else:\\n\",\n    \"        st.dataframe(sop_hits[[\\\"sop_id\\\",\\\"title\\\",\\\"version\\\",\\\"status\\\"]], use_container_width=True, height=220)\\n\",\n    \"        pick = st.selectbox(\\\"Open SOP\\\", [\\\"\\\"] + sop_hits[\\\"sop_id\\\"].astype(str).tolist())\\n\",\n    \"        if pick:\\n\",\n    \"            row = sop_hits[sop_hits[\\\"sop_id\\\"].astype(str)==pick].iloc[0]\\n\",\n    \"            pdf = row.get(\\\"pdf_path\\\",\\\"\\\")\\n\",\n    \"            if pdf: st.write(\\\"PDF path:\\\", pdf)\\n\",\n    \"            if \\\"checklist\\\" in sop_hits.columns and isinstance(row.get(\\\"checklist\\\", None), str) and row[\\\"checklist\\\"].strip():\\n\",\n    \"                st.subheader(\\\"Checklist\\\")\\n\",\n    \"                steps = [s.strip() for s in row[\\\"checklist\\\"].split(\\\"|\\\") if s.strip()]\\n\",\n    \"                completed = []\\n\",\n    \"                for i, step in enumerate(steps, 1):\\n\",\n    \"                    if st.checkbox(f\\\"{i}. {step}\\\", key=f\\\"sop_{pick}_{i}\\\"):\\n\",\n    \"                        completed.append(i)\\n\",\n    \"                st.caption(f\\\"Completed {len(completed)}/{len(steps)} steps\\\")\\n\",\n    \"    \\n\",\n    \"    # V8: Enhanced Patient Monitoring (Phase 1 + 2)\\n\",\n    \"    st.header(\\\"\ud83d\udc65 Patient Monitoring\\\")\\n\",\n    \"    state = get_state()\\n\",\n    \"    \\n\",\n    \"    # Register patient with lingering monitor\\n\",\n    \"    if hasattr(state, 'patient_id') and state.patient_id:\\n\",\n    \"        LINGERING_MONITOR.register_patient(state.patient_id, state)\\n\",\n    \"    \\n\",\n    \"    # Check lingering patients\\n\",\n    \"    lingering_alerts = LINGERING_MONITOR.check_lingering_patients()\\n\",\n    \"    if lingering_alerts:\\n\",\n    \"        st.warning(f\\\"\u26a0\ufe0f {len(lingering_alerts)} lingering patient alerts\\\")\\n\",\n    \"        for alert in lingering_alerts:\\n\",\n    \"            st.error(f\\\"Patient {alert['patient_id']}: {alert['severity']} - {alert['since_vitals_min']:.0f} min since vitals\\\")\\n\",\n    \"    \\n\",\n    \"    if hasattr(state,\\\"feature_dict\\\"):\\n\",\n    \"        feats = state.feature_dict(); since_v = feats.get(\\\"since_vitals_min\\\", None)\\n\",\n    \"        if since_v is not None:\\n\",\n    \"            if since_v > 120: st.error(f\\\"Lingering patient: since_vitals_min={since_v:.0f} > 120\\\")\\n\",\n    \"            else: st.success(f\\\"Vitals recently checked: {since_v:.0f} min\\\")\\n\",\n    \"    if st.button(\\\"Mark vitals now\\\") and hasattr(state,\\\"touch_now\\\"):\\n\",\n    \"        state.touch_now(pd.Timestamp.utcnow()); st.success(\\\"Vitals timestamp updated.\\\")\\n\",\n    \"    \\n\",\n    \"    # V8: Actions & Critic (Enhanced)\\n\",\n    \"    st.header(\\\"\u26a1 Actions & Clinical Decision Support\\\")\\n\",\n    \"    actions = get_actions(state)\\n\",\n    \"    if not actions: st.info(\\\"No actions available.\\\"); return\\n\",\n    \"    p, benefit, burden = critic.score(state, actions)\\n\",\n    \"    import pandas as pd, numpy as np\\n\",\n    \"    view = pd.DataFrame({\\n\",\n    \"        \\\"id\\\":[a.get(\\\"id\\\") for a in actions],\\n\",\n    \"        \\\"label\\\":[a.get(\\\"label\\\") for a in actions],\\n\",\n    \"        \\\"p_accept\\\":np.round(p,3),\\n\",\n    \"        \\\"benefit\\\":np.round(benefit,3),\\n\",\n    \"        \\\"burden\\\":np.round(burden,3)\\n\",\n    \"    }).sort_values([\\\"p_accept\\\",\\\"benefit\\\"], ascending=[False, False])\\n\",\n    \"    st.dataframe(view, use_container_width=True, height=240)\\n\",\n    \"    \\n\",\n    \"    # V8: Phase 2 Clinical Systems Display (when enabled)\\n\",\n    \"    if RUN_PIPELINE and RESULTS_NOTIFIER:\\n\",\n    \"        with st.expander(\\\"\ud83d\udd2c Phase 2 Clinical Systems\\\", expanded=False):\\n\",\n    \"            st.info(\\\"HL7v2 processing, risk scores, and clinical logic active\\\")\\n\",\n    \"            st.write(\\\"ResultsNotifier callback count:\\\", len(RESULTS_NOTIFIER.callbacks))\\n\",\n    \"            if hasattr(state, 'chest_pain') and state.chest_pain:\\n\",\n    \"                st.success(\\\"Chest pain detected - clinical protocols active\\\")\\n\",\n    \"    \\n\",\n    \"    # PHASE 1: Equipment Movement Analytics (Priority)\\n\",\n    \"    st.header(\\\"\ud83d\udcca Equipment Movement Analytics\\\")\\n\",\n    \"    stats = tracker.movement_stats(); per_eq = stats[\\\"moves_per_equipment\\\"]; routes = stats[\\\"routes\\\"]\\n\",\n    \"    if per_eq.empty: st.info(\\\"No movement data yet.\\\")\\n\",\n    \"    else:\\n\",\n    \"        st.subheader(\\\"Moves per equipment\\\"); st.dataframe(per_eq, use_container_width=True, height=240)\\n\",\n    \"        st.subheader(\\\"Top routes\\\"); st.dataframe(routes, use_container_width=True, height=200)\\n\",\n    \"\\n\",\n    \"print(\\\"\u2705 Enhanced UI system ready (Phase 1 priority, Phase 2 integrated)\\\")\"\n   ]\n  },\n  {\n   \"cell_type\": \"code\",\n   \"execution_count\": null,\n   \"id\": \"seed_data\",\n   \"metadata\": {},\n   \"outputs\": [],\n   \"source\": [\n    \"# PHASE 1: SEED DATA (PRESERVED FROM v6)\\n\",\n    \"import pandas as pd\\n\",\n    \"from pathlib import Path\\n\",\n    \"E = Path(CONFIG[\\\"EQUIPMENT_STATUS_PATH\\\"])\\n\",\n    \"if not E.exists():\\n\",\n    \"    pd.DataFrame([\\n\",\n    \"        {\\\"equip_id\\\":\\\"pump-001\\\",\\\"name\\\":\\\"IV Pump\\\",\\\"location\\\":\\\"A1\\\",\\\"status\\\":\\\"ready\\\",\\\"last_seen\\\":pd.Timestamp.utcnow().isoformat(),\\\"battery\\\":0.9,\\\"confidence\\\":0.95},\\n\",\n    \"        {\\\"equip_id\\\":\\\"defib-002\\\",\\\"name\\\":\\\"Defibrillator\\\",\\\"location\\\":\\\"B2\\\",\\\"status\\\":\\\"ready\\\",\\\"last_seen\\\":pd.Timestamp.utcnow().isoformat(),\\\"battery\\\":0.8,\\\"confidence\\\":0.90},\\n\",\n    \"        {\\\"equip_id\\\":\\\"us-003\\\",\\\"name\\\":\\\"Ultrasound\\\",\\\"location\\\":\\\"C1\\\",\\\"status\\\":\\\"ready\\\",\\\"last_seen\\\":pd.Timestamp.utcnow().isoformat(),\\\"battery\\\":0.7,\\\"confidence\\\":0.85},\\n\",\n    \"        {\\\"equip_id\\\":\\\"wheelchair-004\\\",\\\"name\\\":\\\"Wheelchair\\\",\\\"location\\\":\\\"D2\\\",\\\"status\\\":\\\"in_use\\\",\\\"last_seen\\\":pd.Timestamp.utcnow().isoformat(),\\\"battery\\\":None,\\\"confidence\\\":0.95},\\n\",\n    \"    ]).to_csv(E, index=False)\\n\",\n    \"M = Path(CONFIG[\\\"EQUIPMENT_MOVES_LOG_PATH\\\"])\\n\",\n    \"if not M.exists(): pd.DataFrame(columns=[\\\"equip_id\\\",\\\"from\\\",\\\"to\\\",\\\"ts\\\"]).to_csv(M, index=False)\\n\",\n    \"S = Path(CONFIG[\\\"SOP_REGISTRY_PATH\\\"])\\n\",\n    \"if not S.exists():\\n\",\n    \"    sop_dir = Path(CONFIG[\\\"DATA_ROOT\\\"]) / \\\"sop_pdfs\\\"; sop_dir.mkdir(parents=True, exist_ok=True)\\n\",\n    \"    for i in range(1,6): (sop_dir / f\\\"SOP_{i:02d}.pdf\\\").write_bytes(b\\\"%PDF-1.4\\\\n% placeholder\\\\n\\\")\\n\",\n    \"    pd.DataFrame([\\n\",\n    \"        {\\\"sop_id\\\":\\\"SOP_01\\\",\\\"title\\\":\\\"Chest Pain Triage\\\",\\\"pdf_path\\\":str(sop_dir/\\\"SOP_01.pdf\\\"),\\\"version\\\":\\\"1.0\\\",\\\"status\\\":\\\"active\\\",\\\"keywords\\\":\\\"chest pain|ecg|troponin\\\",\\\"checklist\\\":\\\"Open SOP|Order ECG|Record troponin|Reassess vitals\\\"},\\n\",\n    \"        {\\\"sop_id\\\":\\\"SOP_02\\\",\\\"title\\\":\\\"Sepsis Initial Bundle\\\",\\\"pdf_path\\\":str(sop_dir/\\\"SOP_02.pdf\\\"),\\\"version\\\":\\\"1.0\\\",\\\"status\\\":\\\"active\\\",\\\"keywords\\\":\\\"sepsis|qsofa|fluids\\\",\\\"checklist\\\":\\\"Open SOP|Order labs|Start fluids|Antibiotics within 1h\\\"},\\n\",\n    \"        {\\\"sop_id\\\":\\\"SOP_03\\\",\\\"title\\\":\\\"Stroke Code\\\",\\\"pdf_path\\\":str(sop_dir/\\\"SOP_03.pdf\\\"),\\\"version\\\":\\\"1.0\\\",\\\"status\\\":\\\"active\\\",\\\"keywords\\\":\\\"stroke|nihs|ct\\\",\\\"checklist\\\":\\\"Open SOP|CT head|Neurology consult|Thrombolysis criteria\\\"},\\n\",\n    \"        {\\\"sop_id\\\":\\\"SOP_04\\\",\\\"title\\\":\\\"STEMI Fast Track\\\",\\\"pdf_path\\\":str(sop_dir/\\\"SOP_04.pdf\\\"),\\\"version\\\":\\\"1.0\\\",\\\"status\\\":\\\"active\\\",\\\"keywords\\\":\\\"stemi|ecg|cardiology|cath lab\\\",\\\"checklist\\\":\\\"Open SOP|ECG immediate|Page cardiology|Cath lab activation\\\"},\\n\",\n    \"        {\\\"sop_id\\\":\\\"SOP_05\\\",\\\"title\\\":\\\"Equipment Location Update\\\",\\\"pdf_path\\\":str(sop_dir/\\\"SOP_05.pdf\\\"),\\\"version\\\":\\\"1.0\\\",\\\"status\\\":\\\"active\\\",\\\"keywords\\\":\\\"equipment|qr|tracking|location\\\",\\\"checklist\\\":\\\"Scan QR code|Update location|Verify status|Log timestamp\\\"},\\n\",\n    \"    ]).to_csv(S, index=False)\\n\",\n    \"print(\\\"\u2705 Enhanced seed data ready (Phase 1 priority equipment + SOPs)\\\")\"\n   ]\n  },\n  {\n   \"cell_type\": \"code\",\n   \"execution_count\": null,\n   \"id\": \"smoke_tests\",\n   \"metadata\": {},\n   \"outputs\": [],\n   \"source\": [\n    \"# V8 INTEGRATION: SMOKE TESTS (ENHANCED)\\n\",\n    \"import pandas as pd, numpy as np\\n\",\n    \"\\n\",\n    \"print(\\\"\ud83e\uddea Running ED Pipeline v8 smoke tests...\\\")\\n\",\n    \"\\n\",\n    \"# Test 1: Core WorkflowState\\n\",\n    \"s=WorkflowState(role=\\\"nurse\\\", patient_id=\\\"TEST_001\\\", chest_pain=True)\\n\",\n    \"getattr(s,\\\"touch_now\\\",lambda *_:None)(pd.Timestamp.utcnow())\\n\",\n    \"assert hasattr(s, 'feature_dict'), \\\"WorkflowState missing feature_dict\\\"\\n\",\n    \"features = s.feature_dict()\\n\",\n    \"assert 'equipment_tracking_active' in features, \\\"Missing operational features\\\"\\n\",\n    \"print(\\\"\u2705 WorkflowState enhanced features working\\\")\\n\",\n    \"\\n\",\n    \"# Test 2: TinyCritics compatibility\\n\",\n    \"tc=TinyCritics(); p,b,u=tc.score(s,[{\\\"id\\\":\\\"reassess_vitals\\\",\\\"label\\\":\\\"Reassess vitals\\\"},{\\\"id\\\":\\\"order_ecg\\\",\\\"label\\\":\\\"Order ECG\\\"}])\\n\",\n    \"assert len(p)==2 and (0<=p).all() and (p<=1).all(), \\\"TinyCritics scoring failed\\\"\\n\",\n    \"print(\\\"\u2705 TinyCritics compatibility maintained\\\")\\n\",\n    \"\\n\",\n    \"# Test 3: Phase 1 Equipment Tracking\\n\",\n    \"from pathlib import Path\\n\",\n    \"t=TrackerService.from_config(CONFIG)\\n\",\n    \"eq_status=t.equipment_status(); assert not eq_status.empty, \\\"Equipment status empty\\\"\\n\",\n    \"t.log_move(\\\"pump-001\\\",\\\"A1\\\",\\\"B2\\\"); assert Path(CONFIG[\\\"EQUIPMENT_MOVES_LOG_PATH\\\"]).exists(), \\\"Moves log not created\\\"\\n\",\n    \"q=t.make_qr(\\\"v8test\\\"); assert isinstance(q,str) and len(q)>0, \\\"QR generation failed\\\"\\n\",\n    \"print(\\\"\u2705 Equipment tracking system working\\\")\\n\",\n    \"\\n\",\n    \"# Test 4: SOP System\\n\",\n    \"df_sop=t.sop_table(); print(f\\\"\u2705 SOP registry loaded: {len(df_sop)} SOPs\\\")\\n\",\n    \"sop_search = t.search_sop(\\\"chest\\\"); assert not sop_search.empty, \\\"SOP search failed\\\"\\n\",\n    \"print(\\\"\u2705 SOP search system working\\\")\\n\",\n    \"\\n\",\n    \"# Test 5: Lingering Patient Monitor\\n\",\n    \"LINGERING_MONITOR.register_patient(\\\"TEST_001\\\", s)\\n\",\n    \"stats = LINGERING_MONITOR.get_summary_stats()\\n\",\n    \"assert stats['total_patients'] > 0, \\\"Lingering monitor registration failed\\\"\\n\",\n    \"print(\\\"\u2705 Lingering patient monitoring working\\\")\\n\",\n    \"\\n\",\n    \"# Test 6: Phase 2 Clinical Systems (if enabled)\\n\",\n    \"if RUN_PIPELINE and RESULTS_NOTIFIER:\\n\",\n    \"    assert len(RESULTS_NOTIFIER.callbacks) > 0, \\\"ResultsNotifier callbacks not registered\\\"\\n\",\n    \"    # Test corrected troponin logic\\n\",\n    \"    assert RESULTS_NOTIFIER._get_troponin_delta_threshold(13) == 0.50, \\\"Troponin <14 should be 50%\\\"\\n\",\n    \"    assert RESULTS_NOTIFIER._get_troponin_delta_threshold(25) == 0.20, \\\"Troponin 15-50 should be 20%\\\"\\n\",\n    \"    assert RESULTS_NOTIFIER._get_troponin_delta_threshold(55) == 0.50, \\\"Troponin >51 should be 50%\\\"\\n\",\n    \"    print(\\\"\u2705 Phase 2 clinical systems active with corrected troponin logic\\\")\\n\",\n    \"else:\\n\",\n    \"    print(\\\"\u23f8\ufe0f Phase 2 clinical systems disabled (as expected)\\\")\\n\",\n    \"\\n\",\n    \"# Test 7: Enhanced Skills\\n\",\n    \"candidates = generate_candidates(s)\\n\",\n    \"assert len(candidates) > 0, \\\"No action candidates generated\\\"\\n\",\n    \"has_operational = any('EQUIPMENT' in c.get('action', '') or 'SOP' in c.get('action', '') for c in candidates)\\n\",\n    \"print(f\\\"\u2705 Action generation working: {len(candidates)} candidates (operational skills included: {has_operational})\\\")\\n\",\n    \"\\n\",\n    \"print(\\\"\\\\n\ud83c\udf89 ED Pipeline v8 SMOKE TESTS PASSED\\\")\\n\",\n    \"print(\\\"\\\\n\ud83d\udccb System Status Summary:\\\")\\n\",\n    \"print(f\\\"   Phase 1 (Operational): \u2705 Equipment tracking, SOP access, lingering monitoring\\\")\\n\",\n    \"print(f\\\"   Phase 2 (Clinical): {'\u2705 Active' if RUN_PIPELINE else '\u23f8\ufe0f Disabled'} - HL7v2, risk scores, clinical logic\\\")\\n\",\n    \"print(f\\\"   Integration: \u2705 All systems working together\\\")\\n\",\n    \"print(f\\\"   Priority: \u2705 Operational tools primary, clinical systems optional\\\")\\n\",\n    \"print(\\\"\\\\n\ud83d\ude80 Ready for deployment!\\\")\"\n   ]\n  },\n  {\n   \"cell_type\": \"code\",\n   \"execution_count\": null,\n   \"id\": \"main_execution\",\n   \"metadata\": {},\n   \"outputs\": [],\n   \"source\": [\n    \"# V8 LAUNCH: MAIN EXECUTION\\n\",\n    \"tracker = TrackerService.from_config(CONFIG)\\n\",\n    \"\\n\",\n    \"def _get_state():\\n\",\n    \"    s = WorkflowState(role=\\\"nurse\\\", patient_id=\\\"PATIENT_001\\\", chest_pain=True)\\n\",\n    \"    if hasattr(s,\\\"touch_now\\\"): s.touch_now(pd.Timestamp.utcnow())\\n\",\n    \"    return s\\n\",\n    \"\\n\",\n    \"def _get_actions(s):\\n\",\n    \"    # Phase 1 operational actions (priority)\\n\",\n    \"    actions = [\\n\",\n    \"        {\\\"id\\\":\\\"reassess_vitals\\\",\\\"label\\\":\\\"Reassess vitals (Phase 1)\\\"},\\n\",\n    \"        {\\\"id\\\":\\\"check_equipment\\\",\\\"label\\\":\\\"Check equipment locations (Phase 1)\\\"},\\n\",\n    \"        {\\\"id\\\":\\\"access_chest_pain_sop\\\",\\\"label\\\":\\\"Access chest pain SOP (Phase 1)\\\"},\\n\",\n    \"    ]\\n\",\n    \"    \\n\",\n    \"    # Add clinical actions if Phase 2 enabled\\n\",\n    \"    if RUN_PIPELINE:\\n\",\n    \"        actions.extend([\\n\",\n    \"            {\\\"id\\\":\\\"order_ecg\\\",\\\"label\\\":\\\"Order ECG (Phase 2)\\\"},\\n\",\n    \"            {\\\"id\\\":\\\"troponin_protocol\\\",\\\"label\\\":\\\"Troponin protocol (Phase 2)\\\"},\\n\",\n    \"        ])\\n\",\n    \"    \\n\",\n    \"    return actions\\n\",\n    \"\\n\",\n    \"if CONFIG[\\\"RUN_UI\\\"]:\\n\",\n    \"    print(\\\"\ud83d\ude80 Launching ED Pipeline v8 UI...\\\")\\n\",\n    \"    print(\\\"   Phase 1: Equipment tracking, SOP access, lingering monitoring\\\")\\n\",\n    \"    if RUN_PIPELINE:\\n\",\n    \"        print(\\\"   Phase 2: Clinical systems active\\\")\\n\",\n    \"    else:\\n\",\n    \"        print(\\\"   Phase 2: Clinical systems disabled (set RUN_PIPELINE=True to enable)\\\")\\n\",\n    \"    run_ui(tracker=tracker, get_state=_get_state, get_actions=_get_actions, critic=TinyCritics())\\n\",\n    \"else:\\n\",\n    \"    print(\\\"\\\\n\ud83c\udfe5 ED Pipeline v8 Ready\\\")\\n\",\n    \"    print(\\\"=\\\"*50)\\n\",\n    \"    print(\\\"\ud83d\udd27 Phase 1 (OPERATIONAL - PRIORITY):\\\")\\n\",\n    \"    print(\\\"   \u2705 Equipment tracking system\\\")\\n\",\n    \"    print(\\\"   \u2705 QR code generation & scanning\\\")\\n\",\n    \"    print(\\\"   \u2705 SOP quick access system\\\")\\n\",\n    \"    print(\\\"   \u2705 Lingering patient monitoring\\\")\\n\",\n    \"    print(\\\"   \u2705 Real-time dashboard\\\")\\n\",\n    \"    print(\\\"\\\")\\n\",\n    \"    print(\\\"\ud83d\udd2c Phase 2 (CLINICAL - OPTIONAL):\\\")\\n\",\n    \"    if RUN_PIPELINE:\\n\",\n    \"        print(\\\"   \u2705 HL7v2 results processing\\\")\\n\",\n    \"        print(\\\"   \u2705 Risk score calculations\\\")\\n\",\n    \"        print(\\\"   \u2705 Clinical decision support\\\")\\n\",\n    \"    else:\\n\",\n    \"        print(\\\"   \u23f8\ufe0f Disabled (set CONFIG['RUN_PIPELINE']=True to enable)\\\")\\n\",\n    \"    print(\\\"\\\")\\n\",\n    \"    print(\\\"\ud83d\udca1 To launch UI: Set CONFIG['RUN_UI']=True\\\")\\n\",\n    \"    print(\\\"=\\\"*50)\"\n   ]\n  }\n ],\n \"metadata\": {\n  \"kernelspec\": {\n   \"display_name\": \"Python 3\",\n   \"language\": \"python\",\n   \"name\": \"python3\"\n  },\n  \"language_info\": {\n   \"name\": \"python\",\n   \"version\": \"3.11.0\",\n   \"mimetype\": \"text/x-python\",\n   \"file_extension\": \".py\",\n   \"pygments_lexer\": \"ipython3\",\n   \"nbconvert_exporter\": \"python\"\n  }\n },\n \"nbformat\": 4,\n \"nbformat_minor\": 5\n}"
EMBED_PATH.write_text(ED_V8_SOURCE)
print('Wrote canonical to', EMBED_PATH)
print('sha256=', hashlib.sha256(ED_V8_SOURCE.encode('utf-8')).hexdigest())


In [ ]:
from pathlib import Path
import runpy
PHASE1_GATE = False
mod = runpy.run_path('./ed_pipeline_v8.py')
WorkflowState = mod.get('WorkflowState')
TinyCritics = mod.get('TinyCritics')
CONFIG = mod.get('CONFIG')
PHASE1_GATE = all(v is not None for v in [WorkflowState, TinyCritics, CONFIG])
print('Core symbols present:', PHASE1_GATE)
PHASE1_GATE


In [ ]:
if not PHASE1_GATE:
    raise SystemExit('HARD GATE: missing canonical module / core symbols. Fix before proceeding.')
_WS = globals().get('WorkflowState')
assert _WS is not None, 'WorkflowState must be loaded.'
if not hasattr(_WS, 'update_state_from_event'):
    def _update_state_from_event(self, event: dict):
        for candidate in ('apply_event','update_from_event','update_from_dict','update'):
            fn = getattr(self, candidate, None)
            if callable(fn): return fn(event)
        backing = getattr(self, 'state', None)
        if backing is None: backing = {}; setattr(self, 'state', backing)
        for k,v in (event or {}).items():
            parts = str(k).split('.')
            d = backing
            for p in parts[:-1]:
                if p not in d or not isinstance(d[p], dict): d[p] = {}
                d = d[p]
            d[parts[-1]] = v
        return self
    setattr(_WS, 'update_state_from_event', _update_state_from_event)
_=_WS(role='nurse').update_state_from_event({'age':60,'vitals.sbp':95}); print('OK: update_state_from_event present')


In [ ]:
import sys, types, pandas as pd, numpy as np
from pathlib import Path
assert 'CONFIG' in globals(), 'CONFIG must be defined.'
def _cfg(CONFIG, key, default=None):
    try: return CONFIG.get(key, default)
    except Exception: return getattr(CONFIG, key, default) if hasattr(CONFIG, key) else default
def _ensure_parent(p: Path): p=Path(p); p.parent.mkdir(parents=True, exist_ok=True)
def _utcnow_iso(): return pd.Timestamp.utcnow().isoformat()
class EquipmentRepository:
    def __init__(self, status_csv: Path):
        self.status_csv = Path(status_csv); _ensure_parent(self.status_csv)
        if not self.status_csv.exists():
            pd.DataFrame(columns=['equip_id','name','location','status','last_seen','battery','confidence']).to_csv(self.status_csv, index=False)
    def read(self) -> pd.DataFrame:
        try: df = pd.read_csv(self.status_csv)
        except Exception: df = pd.DataFrame(columns=['equip_id','name','location','status','last_seen','battery','confidence'])
        for c in ['equip_id','name','location','status','last_seen','battery','confidence']:
            if c not in df.columns: df[c] = np.nan
        df['equip_id'] = df['equip_id'].astype(str)
        return df[['equip_id','name','location','status','last_seen','battery','confidence']]
    def upsert(self, rec: dict) -> None:
        df = self.read(); eqid = str(rec.get('equip_id',''))
        if (df['equip_id'] == eqid).any():
            idx = df.index[df['equip_id'] == eqid][0]
            for k in ['name','location','status','last_seen','battery','confidence']:
                if k in df.columns:
                    df.loc[idx, k] = rec.get(k, df.loc[idx, k])
        else:
            df = pd.concat([df, pd.DataFrame([rec])], ignore_index=True)
        df.to_csv(self.status_csv, index=False)
class MovesLogRepository:
    def __init__(self, moves_csv: Path):
        self.moves_csv = Path(moves_csv); _ensure_parent(self.moves_csv)
        if not self.moves_csv.exists():
            pd.DataFrame(columns=['equip_id','from','to','ts']).to_csv(self.moves_csv, index=False)
    def append(self, equip_id: str, loc_from: str, loc_to: str, ts_iso: str) -> None:
        row = pd.DataFrame([{'equip_id': str(equip_id), 'from': loc_from, 'to': loc_to, 'ts': ts_iso}])
        try:
            prev = pd.read_csv(self.moves_csv); df = pd.concat([prev, row], ignore_index=True)
        except Exception:
            df = row
        df.to_csv(self.moves_csv, index=False)
    def read(self) -> pd.DataFrame:
        try: return pd.read_csv(self.moves_csv)
        except Exception: return pd.DataFrame(columns=['equip_id','from','to','ts'])
class SOPRegistry:
    def __init__(self, sop_csv: Path):
        self.sop_csv = Path(sop_csv); _ensure_parent(self.sop_csv)
        if not self.sop_csv.exists():
            pd.DataFrame([
                {'sop_id':'sop-triage','title':'ED Triage','url':'about:blank','status':'active'},
                {'sop_id':'sop-ecg','title':'ECG Acquisition','url':'about:blank','status':'active'},
                {'sop_id':'sop-sepsis','title':'Sepsis Bundle','url':'about:blank','status':'active'},
            ]).to_csv(self.sop_csv, index=False)
    def read(self):
        try: return pd.read_csv(self.sop_csv)
        except Exception: return None
class QRService:
    def __init__(self, out_dir: Path):
        self.out_dir = Path(out_dir); self.out_dir.mkdir(parents=True, exist_ok=True)
    def make(self, payload: str) -> str:
        try:
            import qrcode
            img = qrcode.make(payload)
            p = self.out_dir / f"qr_{int(pd.Timestamp.utcnow().timestamp())}.png"; img.save(p); return str(p)
        except Exception:
            p = self.out_dir / f"qr_{int(pd.Timestamp.utcnow().timestamp())}.txt"; p.write_text(payload); return str(p)
    def decode(self, path: str):
        try:
            from PIL import Image; from pyzbar.pyzbar import decode as _decode
            res = _decode(Image.open(path));
            if res: return res[0].data.decode('utf-8', errors='ignore')
        except Exception: pass
        try:
            p = Path(path)
            if p.suffix.lower()=='.txt': return p.read_text()
        except Exception: pass
        return None
class TrackerService:
    def __init__(self, equipment_repo: 'EquipmentRepository', moves_repo: 'MovesLogRepository', sop_registry: 'SOPRegistry', qr: 'QRService', config):
        self.equipment_repo=equipment_repo; self.moves_repo=moves_repo; self.sop_registry=sop_registry; self.qr=qr; self.CONFIG=config
    @classmethod
    def from_config(cls, CONFIG):
        return cls(
            EquipmentRepository(Path(_cfg(CONFIG,'EQUIPMENT_STATUS_PATH'))),
            MovesLogRepository(Path(_cfg(CONFIG,'EQUIPMENT_MOVES_LOG_PATH'))),
            SOPRegistry(Path(_cfg(CONFIG,'SOP_REGISTRY_PATH'))),
            QRService(Path(_cfg(CONFIG,'QR_OUTPUT_DIR'))), CONFIG)
    def equipment_status(self): return self.equipment_repo.read()
    def log_move(self, equip_id: str, loc_from: str, loc_to: str) -> None:
        ts = _utcnow_iso(); df = self.equipment_repo.read(); name = ''
        if 'name' in df.columns and (df['equip_id'].astype(str)==str(equip_id)).any():
            name = df.loc[df['equip_id'].astype(str)==str(equip_id), 'name'].iloc[0]
        rec = {'equip_id': str(equip_id), 'name': name, 'location': loc_to, 'status': 'moved', 'last_seen': ts, 'battery': np.nan, 'confidence': np.nan}
        self.equipment_repo.upsert(rec); self.moves_repo.append(str(equip_id), loc_from or '', loc_to, ts)
    def moves_summary(self):
        log = self.moves_repo.read()
        if log.empty: return {'moves_per_equipment': log, 'routes': log}
        per_eq = log.groupby('equip_id').size().reset_index(name='moves').sort_values('moves', ascending=False)
        routes = log.groupby(['from','to']).size().reset_index(name='count').sort_values('count', ascending=False)
        return {'moves_per_equipment': per_eq, 'routes': routes}
    def sop_table(self): return self.sop_registry.read()
    def search_sop(self, query: str):
        df = self.sop_registry.read().copy(); q = (query or '').strip().lower()
        if df is None or df.empty or not q: return df
        cols = [c for c in ['sop_id','title','keywords','version','status'] if c in df.columns]
        mask = df[cols].astype(str).apply(lambda col: col.str.lower().str.contains(q, na=False)).any(axis=1)
        return df[mask]
import types as _types, sys as _sys
_tracker_mod = _types.ModuleType('tracker_core')
for _name,_obj in {
    'EquipmentRepository': EquipmentRepository,
    'MovesLogRepository': MovesLogRepository,
    'SOPRegistry': SOPRegistry,
    'QRService': QRService,
    'TrackerService': TrackerService,
}.items(): setattr(_tracker_mod,_name,_obj)
_sys.modules['tracker_core'] = _tracker_mod
print('OK: tracker_core (v6 schema) ready.')


In [ ]:
CONFIG['RUN_UI']=False; CONFIG['RUN_PIPELINE']=False
print('OK: CONFIG flags set (RUN_UI=False, RUN_PIPELINE=False)')


In [ ]:
import numpy as np, pandas as pd, os
from pathlib import Path
from tracker_core import TrackerService, QRService, EquipmentRepository, MovesLogRepository, SOPRegistry
if not PHASE1_GATE:
    raise SystemExit('HARD GATE: canonical module missing or incomplete. Upload ed_pipeline_v8.py and re-run.')
os.environ['PYTHONHASHSEED']='0'; np.random.seed(0)
s=WorkflowState(role='nurse')
getattr(s,'touch_now',lambda *_:None)(pd.Timestamp.utcnow())
tc=TinyCritics(); p,b,u=tc.score(s,[{'id':'reassess_vitals','label':'Reassess vitals'},{'id':'order_ecg','label':'Order ECG'}])
assert len(p)==2 and (0<=p).all() and (p<=1).all(), 'TinyCritics bounds failed'
for key in ('EQUIPMENT_STATUS_PATH','EQUIPMENT_MOVES_LOG_PATH','SOP_REGISTRY_PATH','QR_OUTPUT_DIR'):
    Path(CONFIG[key]).parent.mkdir(parents=True, exist_ok=True)
t=TrackerService.from_config(CONFIG)
_=t.equipment_status()
t.log_move('pump-001','A1','B2')
assert Path(CONFIG['EQUIPMENT_MOVES_LOG_PATH']).exists(), 'Moves log path missing'
status=t.equipment_status()
for c in ['equip_id','name','location','status','last_seen','battery','confidence']:
    assert c in status.columns, f'missing column {c}'
print('PHASE1_GATE=PASS')
PHASE1_GATE=True


In [ ]:
from typing import Any, Dict
def refresh_sop_registry(CONFIG: Any, base_url: str='https://sop-notaufnahme.de/sop/') -> Dict[str,Any]:
    try:
        import requests; from bs4 import BeautifulSoup
        return {'ok': True, 'note': 'delegated (network call not executed here)'}
    except Exception as e:
        return {'ok': False, 'reason': f'missing libs: {e}'}
def qr_scan_fallback(payload: str, tracker: 'TrackerService'):
    try:
        parts=dict(kv.split('=',1) for kv in payload.split('&') if '=' in kv)
        eq_id=parts.get('id') or parts.get('equip_id'); to_loc=parts.get('to')
        if eq_id and to_loc:
            tracker.log_move(eq_id, '', to_loc); return {'ok': True, 'equip_id': eq_id, 'to': to_loc}
        return {'ok': False, 'reason': 'missing equip_id/to'}
    except Exception as e:
        return {'ok': False, 'reason': str(e)}
ALERT_THRESHOLDS_MIN={'equipment_overdue':60,'lingering_patient':120}
print('Surfaces present.')


In [ ]:
if not PHASE1_GATE:
    raise SystemExit('HARD GATE failed: Phase 1 must pass before Phase 2.')
if CONFIG.get('RUN_PIPELINE'):
    print('Phase 2 enabled — place guarded assertions here.')
else:
    print("Phase 2 disabled (set CONFIG['RUN_PIPELINE']=True to enable).")
